hybrid7: NVARC fast1 with 8 views (Qwen3-4B TTT, batch 8, merged LoRA) on every task (deadline 6.5 h), then Qwen3.8-27B-FP8 (vLLM 0.19, TP=4, fp8 KV, MTP) with the v7 harness at medium: one episode per task, 64k total tokens, `transform` auto-checked on the training pairs and applied when it matches all of them, up to 2 episodes per unverified task; tasks with NVARC's least confident outputs first; merged by the hybrid6 rule (attempt_2 or empty slots only), verified first.

In [ ]:
import time, os, json
NOTEBOOK_START = time.time()
NVARC_HOURS = 6.5
global_end_time = NOTEBOOK_START + NVARC_HOURS * 3600 - 120
RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
QWEN_KILL_AT = NOTEBOOK_START + (11.5 if RERUN else 2.6) * 3600
if not RERUN:
    os.environ['ARC_TASKS'] = '4c7dc4dd,71e489b6,7b3084d4,8b9c3697,b6f77b65,c4d067a0,dbff022c,eee78d87'  # commit test: NVARC and Qwen on the same 8 dev tasks
for d in ['/kaggle/working/ab/a100', '/kaggle/working/ab/live']:
    os.makedirs(d, exist_ok=True)
print('start', NOTEBOOK_START, '| rerun', RERUN)

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file, save_file

import bz2
import json
import pickle
import traceback
import zlib

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}

# Forward passes made by the DFS, for timing logs.
DFS_STATS = {"calls": 0}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        DFS_STATS["calls"] += 1

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=os.getenv("ARC_MODEL_DIR", "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    data_dir = os.getenv("ARC_DATA_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    if rerun_mode:
        test_path = os.path.join(data_dir, "arc-agi_test_challenges.json")
    else:
        test_path = os.path.join(data_dir, "arc-agi_evaluation_challenges.json")

    arc_test_set = ArcDataset.from_file(test_path)

    work_dir = os.getenv("ARC_WORK_DIR", "/kaggle")
    dir_outputs = os.path.join(work_dir, "inference_outputs")
    os.makedirs(dir_outputs, exist_ok=True)

    n_train_aug = int(os.getenv("ARC_N_TRAIN_AUG", "16"))
    ttt_seed = int(os.getenv("ARC_TTT_SEED", "1"))  # stock: augmentation seed 1, trainer seed 42
    train_args["seed"] = 42 if ttt_seed == 1 else ttt_seed
    train_args["learning_rate"] = float(os.getenv("ARC_LR", "5e-5"))
    train_args["num_train_epochs"] = float(os.getenv("ARC_EPOCHS", "1"))

    # Measure the NLL of the true test output after TTT instead of decoding (TTT-setting search without decode cost).
    nll_only = os.getenv("ARC_NLL_ONLY", "0") == "1" and not rerun_mode
    if nll_only:
        with open(os.path.join(data_dir, "arc-agi_evaluation_solutions.json")) as f:
            truth_solutions = json.load(f)
    task_cap = float(os.getenv("ARC_TASK_CAP", "1200"))
    # Decode views: colour permutations per geometry (stock 2 -> 16 views) and views per forward pass (stock 4).
    n_eval_aug = int(os.getenv("ARC_N_EVAL_AUG", "2"))
    decode_batch = int(os.getenv("ARC_DECODE_BATCH", "4"))
    # Peak VRAM grows with prompt length x batch (about 2.3 MB per prompt token at batch 8), so long prompts fall back to 4.
    big_batch_max_prefix = int(os.getenv("ARC_BIG_BATCH_MAX_PREFIX", "4200"))
    # Fold the LoRA into the base weights for decoding; the base weights are restored exactly after each puzzle.
    merge_lora = os.getenv("ARC_MERGE_LORA", "0") == "1"
    pristine = {}
    if merge_lora:
        for name, module in model.named_modules():
            if hasattr(module, "lora_A") and hasattr(module, "base_layer"):
                pristine[name] = (module, module.base_layer.weight.detach().cpu().clone())

    # When a test output ends up with fewer than two candidate grids, decode its first batch again with a lower
    # probability threshold so that attempt_2 is not wasted (ARC_RELAX_FILL=1).
    relax_fill = os.getenv("ARC_RELAX_FILL", "0") == "1"
    relax_max_score = -np.log(float(os.getenv("ARC_RELAX_P", "0.05")))

    # Stock behaviour only checks the cap between decode batches, so a puzzle can overrun it by a whole DFS.
    # With ARC_STRICT_CAP=1 the cap is also the deadline inside the DFS.
    strict_cap = os.getenv("ARC_STRICT_CAP", "0") == "1"

    # Per-puzzle LoRA weights are kept here (about 1 GB each) so later runs can skip test-time training.
    lora_dir = os.getenv("ARC_LORA_DIR", "")
    if lora_dir:
        os.makedirs(lora_dir, exist_ok=True)

    def solve(key):

        nonlocal model

        info = {}

        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        puzzle_ds = arc_test_set.change_keys([key])

        lora_path = os.path.join(lora_dir, f"{key}.safetensors") if lora_dir else ""

        if lora_path and os.path.exists(lora_path):

            set_peft_model_state_dict(model, load_file(lora_path, device="cuda"), adapter_name="default")
            stats = f"loaded {lora_path}"
            info["lora_loaded"] = True

        else:

            model = FastLanguageModel.for_training(model)


            train_ds = puzzle_ds.augment(n=n_train_aug, shfl_keys=True, seed=ttt_seed)
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer

            if lora_path:
                trained = {k: v.detach().cpu().contiguous() for k, v in get_peft_model_state_dict(model, adapter_name="default").items()}
                save_file(trained, lora_path + ".tmp")
                os.replace(lora_path + ".tmp", lora_path)
                del trained

        model = FastLanguageModel.for_inference(model)

        info["train_seconds"] = time.time() - start_time
        if not isinstance(stats, str):
            info["train_loss"] = float(stats.training_loss)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        if nll_only:
            with torch.inference_mode():
                if merge_lora:
                    model.merge_adapter()
                info["truth_nll"] = {}
                for i, truth in enumerate(truth_solutions[key]):
                    bk = f"{key}_{i}"
                    aug_dataset = ArcDataset(keys=[bk], queries={bk: puzzle_ds_multi.queries.get(bk)}, replies={bk: [truth]})
                    aug_dataset = aug_dataset.augment(seed=zlib.crc32(bk.encode()) % 1024**2)
                    aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                    q = [x["input"] for x in aug_dataset.as_list(formatter)]
                    a = [x["reply"] for x in aug_dataset.as_list(formatter)]
                    info["truth_nll"][bk] = calc_scores(q[:4], a[:4], tokenizer, model) + calc_scores(q[4:], a[4:], tokenizer, model)
            print(f"[Rank {rank}] truth NLL for {key}: " + ", ".join(f"{bk}: {np.mean(v):.2f}" for bk, v in info["truth_nll"].items()))
            return info

        eval_ds = puzzle_ds_multi.augment(n=n_eval_aug, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        stock_batching = n_eval_aug == 2 and decode_batch == 4

        batches = []

        if not stock_batching:
            # Views are batched by prompt length (a forward pass costs about the same for 4 or 8 rows).
            # A transposed view has the same shape as a view rotated by 90 degrees, so they share batches.
            for test_id, subkeys in test_id_to_subkeys.items():
                by_length = defaultdict(list)
                for subkey in subkeys:
                    by_length[len(tokenizer.encode(eval_ds.get(subkey, formatter)["input"]))].append(subkey)
                for length, group in by_length.items():
                    size = decode_batch if length <= big_batch_max_prefix else min(decode_batch, 4)
                    for i in range(0, len(group), size):
                        batches.append(group[i:i+size])

        for test_id, subkeys in (test_id_to_subkeys.items() if stock_batching else []):
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in (test_id_to_subkeys.items() if stock_batching else []):
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        if merge_lora:
            model.merge_adapter()

        with torch.inference_mode():
                
            known_scores = {}

            work = [(subkeys, max_score, "") for subkeys in batches]
            relaxed = set()
            w = 0

            while w < len(work):

                subkeys, batch_max_score, suffix = work[w]
                w += 1

                spend_time = time.time() - start_time
                if spend_time > task_cap or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    info["timeout"] = True
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_start, dfs_calls = time.time(), DFS_STATS["calls"]

                dfs_end_time = min(end_time, start_time + task_cap) if strict_cap else end_time

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, batch_max_score, dfs_end_time)

                info.setdefault("dfs", []).append({
                    "seconds": round(time.time() - dfs_start, 1),
                    "calls": DFS_STATS["calls"] - dfs_calls,
                    "prefix_tokens": max(len(t) for t in tokens),
                    "beams": sum(len(b) for _, b in dfs_result),
                })
                score_start = time.time()

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=zlib.crc32(bk.encode()) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey + suffix), "w") as f:
                            pickle.dump(decoded_result, f)

                info["dfs"][-1]["score_seconds"] = round(time.time() - score_start, 1)

                if relax_fill and w == len(work):
                    # All planned batches are done: queue a relaxed pass for outputs with fewer than two grids.
                    for test_bk in {sk.split(".")[0] for group in batches for sk in group}:
                        n_grids = sum(1 for (b, _) in known_scores if b == test_bk)
                        if n_grids < 2 and test_bk not in relaxed:
                            relaxed.add(test_bk)
                            first = next(group for group in batches if group[0].split(".")[0] == test_bk)
                            work.append((first, relax_max_score, ".relax"))
                            info.setdefault("relaxed", []).append(test_bk)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

        info["n_batches"] = len(batches)
        info["n_scored_grids"] = len(known_scores)
        info["inference_mb"] = memory_allocated
        return info

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        task_start = time.time()

        # One failing puzzle (e.g. OOM) must not take the whole GPU out for the rest of the run.
        try:
            info = solve(key)
            info["status"] = "ok"
        except Exception as e:
            print(f"[Rank {rank}] FAILED {key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            info = {"status": f"error:{type(e).__name__}"}
            gc.collect()
            torch.cuda.empty_cache()

        if merge_lora:
            # Unmerging in bf16 would leave rounding error in the base model, so the saved weights are copied back.
            model.unmerge_adapter()
            with torch.no_grad():
                for module, weight in pristine.values():
                    module.base_layer.weight.copy_(weight)

        info.update(key=key, rank=rank, seconds=time.time() - task_start)
        with open(os.path.join(work_dir, f"timing_rank{rank}.jsonl"), "a") as f:
            f.write(json.dumps(info) + "\n")


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    work_dir = os.getenv("ARC_WORK_DIR", "/kaggle")

    # Fix Unsloth patching issue
    if rank > 0:
        wait_start = time.time()
        # Capped so a worker that died during import cannot block the others forever.
        while not os.path.exists(os.path.join(work_dir, f"worker{rank-1}")) and time.time() - wait_start < 900:
            time.sleep(5)
    
    from arc_solver import worker

    with open(os.path.join(work_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    data_dir = os.getenv("ARC_DATA_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    if rerun_mode:
        test_path = os.path.join(data_dir, "arc-agi_test_challenges.json")
    else:
        test_path = os.path.join(data_dir, "arc-agi_evaluation_challenges.json")

    num_gpus = int(os.getenv("ARC_NUM_GPUS", "4"))

    # Outside the competition rerun only a few puzzles are solved; ARC_TASKS (comma separated) overrides them.
    # The default is the screen8 subset of the dev split (splits/eval_split.json), never holdout puzzles.
    debug_keys = os.getenv("ARC_TASKS", "4c7dc4dd,71e489b6,7b3084d4,8b9c3697,b6f77b65,c4d067a0,dbff022c,eee78d87").split(",")

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in debug_keys:
                continue
        queue.put(key)
    for _ in range(num_gpus):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=num_gpus)


In [ ]:
%%writefile /kaggle/working/ab/a100/runner_parts.py
"""Verbatim parts of the hybrid5 Qwen runner (generated by scripts/build_a100_harness.py; do not edit).

source: kaggle/arc2-hybrid5-nvarc-qwen38-medium/arc2-hybrid5-nvarc-qwen38-medium.ipynb  sha256(runner)=9f5ab5a9e263aefe
"""
import hashlib, importlib.util, json, re, sys
from pathlib import Path

RUNTIME_CONFIG = {'schema': 'arc-qwen38-runtime-config-v1', 'model': {'served_name': 'Qwen/Qwen3.8-27B-FP8', 'source_hint': 'qwen3-8-27b-fp8-repacked-v1'}, 'vllm': {'version': '0.19.0', 'wheelhouse_hint': 'vllm019-cp311-cu128-wheelhouse', 'reasoning_parser': 'qwen3', 'tool_parser': 'qwen3_coder'}, 'hardware': {'gpu_count': 4, 'gpu_name_contains': 'L4', 'tensor_parallel_size': 4}, 'server': {'port': 1234, 'context_len': 65536, 'gpu_memory_utilization': 0.9}, 'scheduler': {'concurrency': 16, 'samples_per_query': 2, 'python_tool_max_concurrency': 4, 'progress_interval_s': 60}, 'sampling': {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'max_tokens': 57344, 'reasoning_effort': 'xhigh', 'enable_thinking': True}, 'agent': {'tools_enabled': True, 'max_turns': 10}, 'pass_policies': {'xhigh': {'reasoning_effort': 'xhigh', 'turn_max_tokens': 57344, 'direct_max_tokens': 57344, 'episode_budget_s': 3600, 'initial_predicted_s': 3600}, 'medium': {'reasoning_effort': 'medium', 'turn_max_tokens': 20480, 'direct_max_tokens': 20480, 'episode_budget_s': 1800, 'initial_predicted_s': 1800}}}

AGGREGATOR_SOURCE = '#!/usr/bin/env python3\n"""Aggregate immutable replica candidates into a complete ARC pass@2 file."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport sys\nimport tempfile\nimport traceback\nimport uuid\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\nCOLLECTION_SCHEMA = "arc-qwen38-kaggle-collection-v1"\nCANDIDATE_SCHEMA = "arc-qwen38-kaggle-candidate-v1"\nSHA256_RE = re.compile(r"^[0-9a-f]{64}$")\n\n\ndef _stable_json(value: Any) -> str:\n    return json.dumps(value, sort_keys=True, separators=(",", ":"))\n\n\ndef _digest(value: Any) -> str:\n    return hashlib.sha256(_stable_json(value).encode()).hexdigest()\n\n\ndef _require_sha256(value: Any, *, context: str) -> str:\n    if not isinstance(value, str) or not SHA256_RE.fullmatch(value):\n        raise RuntimeError(f"{context} must be a lowercase SHA-256 digest")\n    return value\n\n\ndef _quarantine_blocking_directory(path: Path) -> Path:\n    """Move a literal destination directory aside without deleting its contents.\n\n    Kaggle work directories can survive interrupted notebook runs. If a stale\n    run leaves a *directory* at ``submission.json``, ``os.replace`` cannot\n    publish the guaranteed floor and the failure happens before the rich-path\n    degradation handler exists. Preserve the debris under a unique sibling and\n    free the required file path. Symlinks are intentionally not followed: an\n    atomic file replace safely replaces the link itself.\n    """\n    if not path.is_dir() or path.is_symlink():\n        raise NotADirectoryError(f"not a blocking literal directory: {path}")\n    for _ in range(32):\n        quarantine = path.with_name(\n            f"{path.name}.blocking-dir.{uuid.uuid4().hex}.quarantine"\n        )\n        try:\n            os.rename(path, quarantine)\n        except FileExistsError:\n            continue\n        return quarantine\n    raise RuntimeError(f"could not allocate quarantine path for {path}")\n\n\ndef _write_json(path: Path, value: Any) -> None:\n    """Atomically publish JSON via a UNIQUE same-directory temp + fsync.\n\n    A fixed ``<name>.tmp`` sibling can be stale work-directory debris (e.g. a\n    leftover *directory* at that exact path), which makes the FIRST\n    guaranteed-floor publication raise ``IsADirectoryError`` and leave no\n    ``submission.json`` at all. A unique ``mkstemp`` temp in the same directory,\n    flushed + fsynced before ``os.replace`` (and a best-effort directory fsync\n    of the rename), makes the publish crash-durable and immune to stale\n    fixed-name debris. If the destination itself is a literal directory, it is\n    quarantined non-destructively only after the replacement payload has been\n    flushed. ``os.replace`` is atomic within one filesystem.\n    """\n    path.parent.mkdir(parents=True, exist_ok=True)\n    data = json.dumps(value, indent=2, sort_keys=True) + "\\n"\n    fd, tmp_name = tempfile.mkstemp(\n        prefix=path.name + ".", suffix=".tmp", dir=str(path.parent))\n    tmp = Path(tmp_name)\n    try:\n        with os.fdopen(fd, "w") as handle:\n            handle.write(data)\n            handle.flush()\n            os.fsync(handle.fileno())\n        if path.is_dir() and not path.is_symlink():\n            _quarantine_blocking_directory(path)\n        os.replace(tmp, path)\n    except BaseException:\n        tmp.unlink(missing_ok=True)\n        raise\n    try:  # durability of the rename itself; failure here does not lose the file\n        dir_fd = os.open(str(path.parent), os.O_RDONLY)\n        try:\n            os.fsync(dir_fd)\n        finally:\n            os.close(dir_fd)\n    except OSError:\n        pass\n\n\ndef load_collection_contract(\n    output_root: Path,\n    challenges_path: Path,\n    *,\n    challenges: dict[str, Any] | None = None,\n    challenge_sha256: str | None = None,\n) -> dict[str, Any] | None:\n    """Load and verify the immutable sweep contract when candidates exist."""\n\n    path = output_root / "collection_contract.json"\n    candidate_paths = sorted((output_root / "candidates").glob("*/*.json"))\n    if not path.is_file():\n        if candidate_paths:\n            raise RuntimeError(\n                "candidate files exist without collection_contract.json; "\n                "refusing to mix unattributed or stale runs"\n            )\n        return None\n    try:\n        contract = json.loads(path.read_text())\n    except (OSError, json.JSONDecodeError) as exc:\n        raise RuntimeError(f"invalid collection contract: {exc}") from exc\n    if contract.get("schema") != COLLECTION_SCHEMA:\n        raise RuntimeError(f"unexpected collection schema: {contract.get(\'schema\')!r}")\n    collection_id = contract.get("collection_id")\n    payload = {key: value for key, value in contract.items() if key != "collection_id"}\n    if collection_id != _digest(payload):\n        raise RuntimeError("collection contract digest does not verify")\n    if challenges is None or challenge_sha256 is None:\n        challenges, challenge_sha256 = load_public_challenges_with_digest(\n            challenges_path\n        )\n    if contract.get("challenge_sha256") != challenge_sha256:\n        raise RuntimeError(\n            "collection contract challenge digest does not match aggregation input"\n        )\n    task_ids = sorted(str(task_id) for task_id in challenges)\n    if contract.get("task_count") != len(task_ids):\n        raise RuntimeError("collection contract task_count does not match challenges")\n    if contract.get("task_ids_sha256") != _digest(task_ids):\n        raise RuntimeError("collection contract task_ids_sha256 does not match challenges")\n    model = contract.get("model")\n    if not isinstance(model, dict):\n        raise RuntimeError("collection contract has no model identity")\n    if model.get("repo_id") != "Qwen/Qwen3.8-27B":\n        raise RuntimeError("collection contract binds the wrong model repository")\n    if model.get("revision") != "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0":\n        raise RuntimeError("collection contract binds the wrong model revision")\n    _require_sha256(model.get("source_digest"), context="model.source_digest")\n    _require_sha256(\n        model.get("source_contract_sha256"),\n        context="model.source_contract_sha256",\n    )\n    _require_sha256(model.get("receipt_sha256"), context="model.receipt_sha256")\n    if model.get("weight_hashes_verified") is not True:\n        raise RuntimeError("collection contract lacks verified model weight hashes")\n    verified_file_count = model.get("verified_file_count")\n    if (\n        isinstance(verified_file_count, bool)\n        or not isinstance(verified_file_count, int)\n        or verified_file_count < 19\n    ):\n        raise RuntimeError("collection contract has an incomplete model file inventory")\n    runtime = contract.get("runtime")\n    if not isinstance(runtime, dict):\n        raise RuntimeError("collection contract has no runtime identity")\n    _require_sha256(\n        runtime.get("receipt_sha256"), context="runtime.receipt_sha256"\n    )\n    schedule = contract.get("schedule")\n    if not isinstance(schedule, dict):\n        raise RuntimeError("collection contract has no schedule")\n    if schedule.get("world_size") != 4:\n        raise RuntimeError("collection contract must use exactly four replicas")\n    for field in ("rollouts", "max_turns"):\n        value = schedule.get(field)\n        if isinstance(value, bool) or not isinstance(value, int) or value < 1:\n            raise RuntimeError(f"collection schedule {field} must be positive")\n    return contract\n\n\ndef validate_grid(grid: Any, *, context: str = "grid") -> None:\n    if not isinstance(grid, list) or not 1 <= len(grid) <= 30:\n        raise ValueError(f"{context}: invalid height")\n    if not all(isinstance(row, list) for row in grid):\n        raise ValueError(f"{context}: rows must be lists")\n    width = len(grid[0])\n    if not 1 <= width <= 30 or any(len(row) != width for row in grid):\n        raise ValueError(f"{context}: invalid rectangle")\n    if any(\n        isinstance(value, bool)\n        or not isinstance(value, int)\n        or not 0 <= value <= 9\n        for row in grid\n        for value in row\n    ):\n        raise ValueError(f"{context}: invalid color")\n\n\ndef _reject_duplicate_object_pairs(\n    pairs: list[tuple[str, Any]],\n) -> dict[str, Any]:\n    value: dict[str, Any] = {}\n    for key, item in pairs:\n        if key in value:\n            raise ValueError(f"duplicate JSON object key: {key!r}")\n        value[key] = item\n    return value\n\n\ndef validate_public_challenges(challenges: Any) -> dict[str, Any]:\n    """Accept only public ARC train labels and unlabeled test inputs."""\n\n    if not isinstance(challenges, dict) or not challenges:\n        raise ValueError("challenge JSON must be a non-empty task map")\n    for task_id, task in challenges.items():\n        if not isinstance(task_id, str) or not task_id:\n            raise ValueError("challenge task IDs must be non-empty strings")\n        context = f"challenge[{task_id!r}]"\n        if not isinstance(task, dict) or set(task) != {"train", "test"}:\n            raise ValueError(\n                f"{context}: task keys must be exactly train/test; "\n                "hidden labels or metadata are forbidden"\n            )\n        for split in ("train", "test"):\n            pairs = task[split]\n            if not isinstance(pairs, list) or not pairs:\n                raise ValueError(f"{context}.{split}: must be a non-empty list")\n            expected_keys = {"input", "output"} if split == "train" else {"input"}\n            for index, pair in enumerate(pairs):\n                pair_context = f"{context}.{split}[{index}]"\n                if not isinstance(pair, dict) or set(pair) != expected_keys:\n                    raise ValueError(\n                        f"{pair_context}: keys must be exactly "\n                        f"{sorted(expected_keys)!r}; hidden test labels are forbidden"\n                    )\n                validate_grid(pair["input"], context=f"{pair_context}.input")\n                if split == "train":\n                    validate_grid(pair["output"], context=f"{pair_context}.output")\n    return challenges\n\n\ndef load_public_challenges_with_digest(\n    path: Path,\n) -> tuple[dict[str, Any], str]:\n    try:\n        raw = path.read_bytes()\n        value = json.loads(\n            raw.decode("utf-8"), object_pairs_hook=_reject_duplicate_object_pairs\n        )\n    except (OSError, UnicodeDecodeError, json.JSONDecodeError, ValueError) as exc:\n        raise RuntimeError(f"invalid public challenge JSON: {exc}") from exc\n    try:\n        challenges = validate_public_challenges(value)\n    except ValueError as exc:\n        raise RuntimeError(f"unsafe public challenge JSON: {exc}") from exc\n    return challenges, hashlib.sha256(raw).hexdigest()\n\n\ndef load_public_challenges(path: Path) -> dict[str, Any]:\n    return load_public_challenges_with_digest(path)[0]\n\n\ndef _valid_outputs(task_id: str, task: dict[str, Any], outputs: Any) -> bool:\n    if not isinstance(outputs, list) or len(outputs) != len(task["test"]):\n        return False\n    try:\n        for index, grid in enumerate(outputs):\n            validate_grid(grid, context=f"{task_id}.candidate[{index}]")\n    except ValueError:\n        return False\n    return True\n\n\ndef _identity(task: dict[str, Any]) -> list[list[list[int]]]:\n    return [pair["input"] for pair in task["test"]]\n\n\ndef _background(task: dict[str, Any]) -> list[list[list[int]]]:\n    counts = Counter(\n        value\n        for pair in task["train"]\n        for row in pair["output"]\n        for value in row\n    )\n    color = counts.most_common(1)[0][0]\n    same_shape = all(\n        (len(pair["input"]), len(pair["input"][0]))\n        == (len(pair["output"]), len(pair["output"][0]))\n        for pair in task["train"]\n    )\n    shapes = {\n        (len(pair["output"]), len(pair["output"][0])) for pair in task["train"]\n    }\n    fixed = next(iter(shapes)) if len(shapes) == 1 else None\n    rows = []\n    for pair in task["test"]:\n        if same_shape or fixed is None:\n            height, width = len(pair["input"]), len(pair["input"][0])\n        else:\n            height, width = fixed\n        rows.append([[color] * width for _ in range(height)])\n    return rows\n\n\ndef _geometric_grid(source: str, grid: list[list[int]]) -> list[list[int]]:\n    if source == "fallback-flip-lr":\n        return [list(reversed(row)) for row in grid]\n    if source == "fallback-flip-ud":\n        return list(reversed([row[:] for row in grid]))\n    if source == "fallback-transpose":\n        return [list(row) for row in zip(*grid)]\n    if source == "fallback-rotate-cw":\n        return [list(row) for row in zip(*reversed(grid))]\n    if source == "fallback-rotate-ccw":\n        return [list(row) for row in reversed(list(zip(*grid)))]\n    if source == "fallback-rotate-180":\n        return [list(reversed(row)) for row in reversed(grid)]\n    raise ValueError(f"unknown geometric fallback: {source}")\n\n\ndef _constant_color(task: dict[str, Any], color: int) -> list[list[list[int]]]:\n    same_shape = all(\n        (len(pair["input"]), len(pair["input"][0]))\n        == (len(pair["output"]), len(pair["output"][0]))\n        for pair in task["train"]\n    )\n    shapes = {\n        (len(pair["output"]), len(pair["output"][0])) for pair in task["train"]\n    }\n    fixed = next(iter(shapes)) if len(shapes) == 1 else None\n    rows = []\n    for pair in task["test"]:\n        if same_shape or fixed is None:\n            height, width = len(pair["input"]), len(pair["input"][0])\n        else:\n            height, width = fixed\n        rows.append([[color] * width for _ in range(height)])\n    return rows\n\n\ndef _fallback_outputs(\n    task: dict[str, Any], source: str\n) -> list[list[list[int]]]:\n    if source == "fallback-identity":\n        return _identity(task)\n    if source == "fallback-background":\n        return _background(task)\n    if source.startswith("fallback-constant-"):\n        return _constant_color(task, int(source.rsplit("-", 1)[-1]))\n    return [_geometric_grid(source, pair["input"]) for pair in task["test"]]\n\n\ndef _fallback_demo_score(\n    task: dict[str, Any], source: str\n) -> tuple[bool, float]:\n    if source == "fallback-identity":\n        matches = [pair["input"] == pair["output"] for pair in task["train"]]\n    elif source == "fallback-background":\n        counts = Counter(\n            value\n            for pair in task["train"]\n            for row in pair["output"]\n            for value in row\n        )\n        color = counts.most_common(1)[0][0]\n        same_shape = all(\n            (len(pair["input"]), len(pair["input"][0]))\n            == (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        )\n        shapes = {\n            (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        }\n        fixed = next(iter(shapes)) if len(shapes) == 1 else None\n        matches = []\n        for pair in task["train"]:\n            if same_shape or fixed is None:\n                height, width = len(pair["input"]), len(pair["input"][0])\n            else:\n                height, width = fixed\n            matches.append([[color] * width for _ in range(height)] == pair["output"])\n    elif source.startswith("fallback-constant-"):\n        color = int(source.rsplit("-", 1)[-1])\n        same_shape = all(\n            (len(pair["input"]), len(pair["input"][0]))\n            == (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        )\n        shapes = {\n            (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        }\n        fixed = next(iter(shapes)) if len(shapes) == 1 else None\n        matches = []\n        for pair in task["train"]:\n            if same_shape or fixed is None:\n                height, width = len(pair["input"]), len(pair["input"][0])\n            else:\n                height, width = fixed\n            matches.append([[color] * width for _ in range(height)] == pair["output"])\n    else:\n        matches = [\n            _geometric_grid(source, pair["input"]) == pair["output"]\n            for pair in task["train"]\n        ]\n    return all(matches), sum(matches) / len(matches)\n\n\ndef _trajectory_id(row: dict[str, Any]) -> tuple[int, int] | None:\n    rank = row.get("rank")\n    rollout = row.get("rollout")\n    if not isinstance(rank, int) or isinstance(rank, bool):\n        return None\n    if not isinstance(rollout, int) or isinstance(rollout, bool):\n        return None\n    return rank, rollout\n\n\ndef _trajectory_candidate_key(\n    row: dict[str, Any],\n) -> tuple[bool, float, int, int, str]:\n    turn = row.get("turn")\n    turn_value = turn if isinstance(turn, int) and not isinstance(turn, bool) else -1\n    return (\n        row.get("visible_pass") is True,\n        float(row.get("demo_score") or 0.0),\n        turn_value,\n        -int(row.get("program_cost") or 0),\n        str(row.get("run_id") or ""),\n    )\n\n\ndef _validate_selection_candidates(\n    task_id: str, task: dict[str, Any], candidates: list[dict[str, Any]]\n) -> None:\n    """Reject malformed in-memory candidates before ranking or clustering."""\n\n    seen_run_ids: set[str] = set()\n    for index, row in enumerate(candidates):\n        context = f"{task_id}.candidates[{index}]"\n        if not isinstance(row, dict):\n            raise RuntimeError(f"{context}: candidate must be an object")\n        if _trajectory_id(row) is None:\n            raise RuntimeError(f"{context}: candidate has no valid rank/rollout identity")\n        turn = row.get("turn")\n        if turn is not None and (\n            isinstance(turn, bool) or not isinstance(turn, int) or turn < 0\n        ):\n            raise RuntimeError(f"{context}: turn must be a non-negative integer")\n        run_id = row.get("run_id")\n        if not isinstance(run_id, str) or not run_id:\n            raise RuntimeError(f"{context}: run_id must be non-empty")\n        if run_id in seen_run_ids:\n            raise RuntimeError(f"{context}: duplicate run_id {run_id!r}")\n        seen_run_ids.add(run_id)\n        if not isinstance(row.get("visible_pass"), bool):\n            raise RuntimeError(f"{context}: visible_pass must be boolean")\n        demo_score = row.get("demo_score")\n        if (\n            isinstance(demo_score, bool)\n            or not isinstance(demo_score, (int, float))\n            or not math.isfinite(float(demo_score))\n            or not 0.0 <= float(demo_score) <= 1.0\n        ):\n            raise RuntimeError(f"{context}: demo_score must be finite in 0..1")\n        program_cost = row.get("program_cost")\n        if (\n            isinstance(program_cost, bool)\n            or not isinstance(program_cost, int)\n            or program_cost < 0\n        ):\n            raise RuntimeError(f"{context}: program_cost must be non-negative")\n        if not _valid_outputs(task_id, task, row.get("test_outputs")):\n            raise RuntimeError(f"{context}: candidate test outputs are invalid")\n\n\ndef _trajectory_representatives(\n    candidates: list[dict[str, Any]],\n) -> list[dict[str, Any]]:\n    """Choose one vote-bearing candidate per independent trajectory."""\n\n    best: dict[tuple[int, int], dict[str, Any]] = {}\n    for row in candidates:\n        trajectory = _trajectory_id(row)\n        if trajectory is None:\n            continue\n        current = best.get(trajectory)\n        if current is None or _trajectory_candidate_key(row) > _trajectory_candidate_key(\n            current\n        ):\n            best[trajectory] = row\n    return [best[key] for key in sorted(best)]\n\n\ndef _selection_key(row: dict[str, Any]) -> tuple[Any, ...]:\n    """Order hypotheses by visible evidence without conflating repairs.\n\n    ``support`` contains only the one vote-bearing state selected from each\n    independent ``(rank, rollout)`` trajectory.  Older repair states remain\n    useful pass@2 hypotheses, but their raw provenance can break ties only\n    after independent support and can never manufacture another vote.\n    """\n\n    return (\n        row["visible_pass"],\n        row["demo_score"],\n        row["visible_pass_support"],\n        row["support"],\n        row["model_derived"],\n        row["raw_visible_pass_trajectory_support"],\n        row["raw_trajectory_support"],\n        row["replica_support"],\n        row["rollout_support"],\n        -row["program_cost"],\n        row["representative_id"],\n    )\n\n\ndef load_candidates(\n    output_root: Path,\n    task_id: str,\n    task: dict[str, Any],\n    *,\n    collection_contract: dict[str, Any] | None = None,\n) -> list[dict[str, Any]]:\n    rows = []\n    directory = output_root / "candidates" / task_id\n    if not directory.is_dir():\n        return rows\n    if collection_contract is None:\n        raise RuntimeError(\n            f"{task_id}: candidate directory exists without a verified collection contract"\n        )\n    collection_id = collection_contract["collection_id"]\n    contract_sha256 = _digest(collection_contract)\n    seen_run_ids: set[str] = set()\n    schedule = collection_contract["schedule"]\n    world_size = int(schedule["world_size"])\n    rollouts = int(schedule["rollouts"])\n    max_turns = int(schedule["max_turns"])\n    for path in sorted(directory.glob("*.json")):\n        try:\n            row = json.loads(path.read_text())\n        except (OSError, json.JSONDecodeError) as exc:\n            raise RuntimeError(f"{path}: invalid candidate JSON: {exc}") from exc\n        if not isinstance(row, dict):\n            raise RuntimeError(f"{path}: candidate must be an object")\n        if row.get("schema") != CANDIDATE_SCHEMA:\n            raise RuntimeError(f"{path}: unexpected candidate schema")\n        declared_record_sha256 = _require_sha256(\n            row.get("candidate_record_sha256"),\n            context=f"{path}: candidate_record_sha256",\n        )\n        record_payload = {\n            key: value\n            for key, value in row.items()\n            if key != "candidate_record_sha256"\n        }\n        if declared_record_sha256 != _digest(record_payload):\n            raise RuntimeError(f"{path}: candidate record digest does not verify")\n        if row.get("collection_id") != collection_id:\n            raise RuntimeError(\n                f"{path}: candidate belongs to a different or unattributed collection"\n            )\n        if row.get("collection_contract_sha256") != contract_sha256:\n            raise RuntimeError(f"{path}: candidate collection-contract digest mismatch")\n        if row.get("challenge_sha256") != collection_contract["challenge_sha256"]:\n            raise RuntimeError(f"{path}: candidate challenge digest mismatch")\n        if row.get("model") != collection_contract["model"]:\n            raise RuntimeError(f"{path}: candidate model identity mismatch")\n        if row.get("runtime_receipt_sha256") != collection_contract["runtime"].get(\n            "receipt_sha256"\n        ):\n            raise RuntimeError(f"{path}: candidate runtime identity mismatch")\n        if row.get("task_id") != task_id:\n            raise RuntimeError(f"{path}: candidate task ID does not match its directory")\n        trajectory = _trajectory_id(row)\n        if trajectory is None:\n            raise RuntimeError(f"{path}: candidate has no valid rank/rollout identity")\n        rank, rollout = trajectory\n        turn = row.get("turn")\n        if not 0 <= rank < world_size:\n            raise RuntimeError(f"{path}: rank is outside the collection world size")\n        if not 0 <= rollout < rollouts:\n            raise RuntimeError(f"{path}: rollout is outside the collection schedule")\n        if isinstance(turn, bool) or not isinstance(turn, int) or not 0 <= turn < max_turns:\n            raise RuntimeError(f"{path}: turn is outside the collection schedule")\n        expected_name = f"rank{rank}-rollout{rollout}-turn{turn}.json"\n        if path.name != expected_name:\n            raise RuntimeError(\n                f"{path}: candidate filename does not match trajectory identity"\n            )\n        expected_run_id = f"{task_id}-rank{rank}-rollout{rollout}-turn{turn}"\n        run_id = row.get("run_id")\n        if run_id != expected_run_id:\n            raise RuntimeError(f"{path}: candidate run_id does not match trajectory")\n        if run_id in seen_run_ids:\n            raise RuntimeError(f"{path}: duplicate candidate run_id {run_id!r}")\n        seen_run_ids.add(run_id)\n        if row.get("source") != "qwen38-nf4-replica":\n            raise RuntimeError(f"{path}: unexpected candidate source")\n        if not isinstance(row.get("visible_pass"), bool):\n            raise RuntimeError(f"{path}: visible_pass must be boolean")\n        demo_score = row.get("demo_score")\n        if (\n            isinstance(demo_score, bool)\n            or not isinstance(demo_score, (int, float))\n            or not math.isfinite(float(demo_score))\n            or not 0.0 <= float(demo_score) <= 1.0\n        ):\n            raise RuntimeError(f"{path}: demo_score must be finite in 0..1")\n        program = row.get("program")\n        if not isinstance(program, str) or not program.strip():\n            raise RuntimeError(f"{path}: program must be non-empty")\n        program_sha256 = _require_sha256(\n            row.get("program_sha256"), context=f"{path}: program_sha256"\n        )\n        if program_sha256 != hashlib.sha256(program.encode()).hexdigest():\n            raise RuntimeError(f"{path}: program_sha256 does not match program bytes")\n        program_cost = row.get("program_cost")\n        if (\n            isinstance(program_cost, bool)\n            or not isinstance(program_cost, int)\n            or program_cost != len(program)\n        ):\n            raise RuntimeError(f"{path}: program_cost does not match program length")\n        if not _valid_outputs(task_id, task, row.get("test_outputs")):\n            raise RuntimeError(f"{path}: candidate test outputs are invalid")\n        test_outputs_sha256 = _require_sha256(\n            row.get("test_outputs_sha256"),\n            context=f"{path}: test_outputs_sha256",\n        )\n        if test_outputs_sha256 != _digest(row["test_outputs"]):\n            raise RuntimeError(\n                f"{path}: test_outputs_sha256 does not match test outputs"\n            )\n        rows.append(row)\n    return rows\n\n\ndef validate_submission_completeness(\n    challenges: dict[str, Any], submission: Any\n) -> None:\n    """Assert a submission is structurally complete and grid-valid.\n\n    The single reusable definition of "a submittable file": exactly the\n    challenge task set, one attempt object per test query, each object exactly\n    {attempt_1, attempt_2}, every attempt a valid ARC grid (1..30 dims, colours\n    0..9), and the two attempts distinct. Reused by :func:`aggregate`, the\n    last-resort fallback path, and the offline robustness harness so the\n    guarantee has one source of truth.\n    """\n\n    if not isinstance(submission, dict):\n        raise ValueError("submission must be a task->rows object")\n    if set(submission) != set(challenges):\n        missing = sorted(set(challenges) - set(submission))\n        extra = sorted(set(submission) - set(challenges))\n        raise ValueError(\n            f"submission task set mismatch (missing={missing}, extra={extra})"\n        )\n    for task_id, task in challenges.items():\n        rows = submission[task_id]\n        if not isinstance(rows, list) or len(rows) != len(task["test"]):\n            raise ValueError(f"{task_id}: wrong submission output count")\n        for index, row in enumerate(rows):\n            if not isinstance(row, dict) or set(row) != {"attempt_1", "attempt_2"}:\n                raise ValueError(f"{task_id}[{index}]: malformed attempt object")\n            for name in ("attempt_1", "attempt_2"):\n                validate_grid(row[name], context=f"{task_id}[{index}].{name}")\n            if row["attempt_1"] == row["attempt_2"]:\n                raise ValueError(f"{task_id}[{index}]: duplicate pass@2 attempts")\n\n\ndef guaranteed_fallback_submission(\n    challenges: dict[str, Any],\n) -> dict[str, list[dict[str, Any]]]:\n    """Build a structurally-complete submission that cannot fail to be valid.\n\n    This is the robustness floor for the BASE: it depends only on the already\n    validated public test *inputs* (never on train outputs, the collection\n    contract, model candidates, or the rich selection machinery), so it can be\n    written to disk before the model even loads and can never itself raise for a\n    validated challenge set. attempt_1 is the test input (identity); attempt_2\n    is the first distinct dihedral transform. If the input is invariant under\n    every such transform, one cell is changed to keep pass@2 from wasting its\n    second slot on an exact duplicate. A better submission from\n    :func:`aggregate` overwrites it on success; if every richer path fails, this\n    remains and the run still submits a valid pass@2 file.\n    """\n\n    submission: dict[str, list[dict[str, Any]]] = {}\n    for task_id, task in challenges.items():\n        rows: list[dict[str, Any]] = []\n        for pair in task["test"]:\n            grid = pair["input"]\n            candidates = [\n                _geometric_grid("fallback-flip-lr", grid),\n                _geometric_grid("fallback-flip-ud", grid),\n                _geometric_grid("fallback-rotate-180", grid),\n                _geometric_grid("fallback-transpose", grid),\n            ]\n            second = next(\n                (candidate for candidate in candidates if candidate != grid),\n                None,\n            )\n            if second is None:\n                second = [row[:] for row in grid]\n                second[0][0] = (second[0][0] + 1) % 10\n            rows.append({"attempt_1": grid, "attempt_2": second})\n        submission[task_id] = rows\n    # Fail loud if this floor is ever not valid: it means the input contract\n    # was violated upstream, which must surface, not be silently shipped.\n    validate_submission_completeness(challenges, submission)\n    return submission\n\n\ndef select_task(\n    task_id: str, task: dict[str, Any], candidates: list[dict[str, Any]]\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    fallback_sources = (\n        "fallback-identity",\n        "fallback-background",\n        "fallback-flip-lr",\n        "fallback-flip-ud",\n        "fallback-transpose",\n        "fallback-rotate-cw",\n        "fallback-rotate-ccw",\n        "fallback-rotate-180",\n    ) + tuple(f"fallback-constant-{color}" for color in range(10))\n    fallback_outputs = {\n        source: _fallback_outputs(task, source) for source in fallback_sources\n    }\n    _validate_selection_candidates(task_id, task, candidates)\n    submission_rows = []\n    query_receipts = []\n    trajectory_candidates = _trajectory_representatives(candidates)\n    for query_index in range(len(task["test"])):\n        # One trajectory can revise its output over several turns. Select its\n        # strongest state before clustering so it cannot vote for competing\n        # hypotheses at the same cutoff.\n        groups: dict[str, list[dict[str, Any]]] = {}\n        for candidate in trajectory_candidates:\n            grid = candidate["test_outputs"][query_index]\n            groups.setdefault(_digest(grid), []).append(candidate)\n        raw_groups: dict[str, list[dict[str, Any]]] = {}\n        for candidate in candidates:\n            grid = candidate["test_outputs"][query_index]\n            raw_groups.setdefault(_digest(grid), []).append(candidate)\n        ranked = []\n        for output_sha256, raw_members in raw_groups.items():\n            members = groups.get(output_sha256, [])\n            # A vote-bearing representative must come from the trajectory\n            # representatives themselves.  Otherwise a superseded state with\n            # the same test output could silently inflate attempt-1\'s visible\n            # score.  Correlated-only clusters have no such member, so their\n            # best raw state is used for attempt-2 ranking only.\n            representative = max(\n                members or raw_members,\n                key=_trajectory_candidate_key,\n            )\n            replicas = sorted(\n                {\n                    int(row["rank"])\n                    for row in members\n                    if isinstance(row.get("rank"), int)\n                }\n            )\n            rollouts = sorted(\n                {\n                    int(row["rollout"])\n                    for row in members\n                    if isinstance(row.get("rollout"), int)\n                }\n            )\n            trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in members\n            }\n            visible_trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in members\n                if row.get("visible_pass") is True\n            }\n            raw_trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in raw_members\n            }\n            raw_visible_trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in raw_members\n                if row.get("visible_pass") is True\n            }\n            correlated_only = not members\n            ranked.append(\n                {\n                    "grid": representative["test_outputs"][query_index],\n                    "output_sha256": output_sha256,\n                    "source": str(\n                        representative.get("source") or "qwen38-nf4-replica"\n                    ),\n                    "representative_id": str(representative.get("run_id") or ""),\n                    "visible_pass": representative.get("visible_pass") is True,\n                    "demo_score": float(representative.get("demo_score") or 0.0),\n                    "program_cost": int(representative.get("program_cost") or 0),\n                    "model_derived": True,\n                    "correlated_only": correlated_only,\n                    "attempt_1_eligible": not correlated_only,\n                    "hypothesis_origin": (\n                        "correlated-repair-state"\n                        if correlated_only\n                        else "independent-trajectory-vote"\n                    ),\n                    # Multiple repair turns within one rollout are correlated\n                    # and count as one vote. Consensus means independent\n                    # rank/rollout trajectories, not raw candidate files.\n                    "support": len(trajectories),\n                    "independent_support": len(trajectories),\n                    "raw_candidate_support": len(raw_members),\n                    "raw_trajectory_support": len(raw_trajectories),\n                    "raw_visible_pass_trajectory_support": len(\n                        raw_visible_trajectories\n                    ),\n                    "visible_pass_support": len(visible_trajectories),\n                    "replica_support": len(replicas),\n                    "rollout_support": len(rollouts),\n                    "supporting_replicas": replicas,\n                    "supporting_rollouts": rollouts,\n                    "supporting_ids": sorted(\n                        str(row.get("run_id") or "") for row in members\n                    ),\n                    "raw_supporting_ids": sorted(\n                        str(row.get("run_id") or "") for row in raw_members\n                    ),\n                }\n            )\n        # Visible evidence is primary.  Among equally fitting programs, the\n        # output with the most independent support wins attempt 1.  Attempt 2\n        # is the strongest distinct cluster, maximizing pass@2 diversity.\n        ranked.sort(key=_selection_key, reverse=True)\n        for source in fallback_sources:\n            grid = fallback_outputs[source][query_index]\n            fallback_pass, fallback_score = _fallback_demo_score(task, source)\n            ranked.append(\n                {\n                    "grid": grid,\n                    "output_sha256": _digest(grid),\n                    "source": source,\n                    "representative_id": source,\n                    "visible_pass": fallback_pass,\n                    "demo_score": fallback_score,\n                    "program_cost": 0,\n                    "model_derived": False,\n                    "correlated_only": False,\n                    "attempt_1_eligible": True,\n                    "hypothesis_origin": "deterministic-fallback",\n                    "support": 0,\n                    "independent_support": 0,\n                    "raw_candidate_support": 0,\n                    "raw_trajectory_support": 0,\n                    "raw_visible_pass_trajectory_support": 0,\n                    "visible_pass_support": 0,\n                    "replica_support": 0,\n                    "rollout_support": 0,\n                    "supporting_replicas": [],\n                    "supporting_rollouts": [],\n                    "supporting_ids": [],\n                    "raw_supporting_ids": [],\n                }\n            )\n        ranked.sort(key=_selection_key, reverse=True)\n\n        # A superseded repair state is deliberately ineligible for attempt 1:\n        # its trajectory cast its sole independent vote for another state.\n        # It remains eligible for attempt 2, where preserving a strong,\n        # distinct model-derived hypothesis is preferable to erasing it and\n        # falling back to an unrelated geometric guess.\n        attempt_1_pool = [row for row in ranked if row["attempt_1_eligible"]]\n        if not attempt_1_pool:\n            raise RuntimeError(\n                f"{task_id}.test[{query_index}]: no attempt-1-eligible hypothesis"\n            )\n        selected = [attempt_1_pool[0]]\n        seen = {attempt_1_pool[0]["output_sha256"]}\n        for row in ranked:\n            if row["output_sha256"] in seen:\n                continue\n            selected.append(row)\n            seen.add(row["output_sha256"])\n            if len(selected) == 2:\n                break\n        if len(selected) != 2:\n            raise RuntimeError(\n                f"{task_id}.test[{query_index}]: fewer than two distinct "\n                "evidence-ranked output hypotheses survived selection"\n            )\n        submission_rows.append(\n            {"attempt_1": selected[0]["grid"], "attempt_2": selected[1]["grid"]}\n        )\n        query_receipts.append(\n            {\n                "query_index": query_index,\n                "candidate_clusters": len(groups),\n                "raw_candidate_clusters": len(raw_groups),\n                "correlated_only_candidate_clusters": len(\n                    set(raw_groups) - set(groups)\n                ),\n                "selected": [\n                    {key: row[key] for key in row if key != "grid"}\n                    for row in selected\n                ],\n            }\n        )\n    return submission_rows, {\n        "candidate_count": len(candidates),\n        "trajectory_candidate_count": len(trajectory_candidates),\n        "visible_pass_candidates": sum(\n            row.get("visible_pass") is True for row in candidates\n        ),\n        "queries": query_receipts,\n    }\n\n\ndef aggregate(\n    challenges_path: Path, output_root: Path, submission_path: Path\n) -> tuple[dict[str, Any], dict[str, Any]]:\n    challenges, challenge_sha256 = load_public_challenges_with_digest(\n        challenges_path\n    )\n    collection_contract = load_collection_contract(\n        output_root,\n        challenges_path,\n        challenges=challenges,\n        challenge_sha256=challenge_sha256,\n    )\n    submission = {}\n    tasks = {}\n    for task_id, task in challenges.items():\n        candidates = load_candidates(\n            output_root,\n            task_id,\n            task,\n            collection_contract=collection_contract,\n        )\n        rows, receipt = select_task(task_id, task, candidates)\n        submission[task_id] = rows\n        tasks[task_id] = receipt\n    validate_submission_completeness(challenges, submission)\n    _write_json(submission_path, submission)\n    selected_sources = Counter(\n        selected["source"]\n        for task in tasks.values()\n        for query in task["queries"]\n        for selected in query["selected"]\n    )\n    receipt = {\n        "schema": "arc-qwen38-kaggle-pass2-aggregation-v1",\n        "observed_utc": datetime.now(timezone.utc).isoformat(),\n        "selection_scope": "per-test-query",\n        "collection_id": (\n            collection_contract["collection_id"] if collection_contract else None\n        ),\n        "collection_contract_sha256": (\n            _digest(collection_contract) if collection_contract else None\n        ),\n        "selection_policy": (\n            "attempt-1 excludes superseded repair states; then visible-pass, "\n            "demo-score, independent visible-pass support, independent "\n            "trajectory support, model-derived provenance, raw trajectory "\n            "provenance, replica/rollout spread, shorter-program; attempt-2 "\n            "is the strongest distinct hypothesis including zero-vote "\n            "superseded repair states"\n        ),\n        "challenge_sha256": challenge_sha256,\n        "submission_sha256": hashlib.sha256(submission_path.read_bytes()).hexdigest(),\n        "task_count": len(challenges),\n        "query_count": sum(len(task["test"]) for task in challenges.values()),\n        "candidate_count": sum(row["candidate_count"] for row in tasks.values()),\n        "tasks_with_candidate": sum(row["candidate_count"] > 0 for row in tasks.values()),\n        "tasks_with_visible_pass_candidate": sum(\n            row["visible_pass_candidates"] > 0 for row in tasks.values()\n        ),\n        "selected_source_counts": dict(sorted(selected_sources.items())),\n        "tasks": tasks,\n        "status": "pass",\n    }\n    _write_json(output_root / "aggregation_receipt.json", receipt)\n    return submission, receipt\n\n\ndef _emit_summary(summary: dict[str, Any]) -> None:\n    print(\n        "QWEN38_PASS2_AGGREGATION=" + json.dumps(summary, sort_keys=True),\n        flush=True,\n    )\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--challenges", type=Path, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--submission", type=Path, required=True)\n    parser.add_argument(\n        "--guarantee-only",\n        action="store_true",\n        help=(\n            "Write only the dependency-free guaranteed-fallback submission and "\n            "exit 0. The kernel calls this first so a valid, complete "\n            "submission exists on disk before the model loads or any candidate "\n            "is produced."\n        ),\n    )\n    args = parser.parse_args()\n\n    # ROBUSTNESS FLOOR: a valid, complete submission must exist on disk no\n    # matter what fails later.  Build it from the validated public inputs and\n    # write it before anything richer is attempted.  If the challenges\n    # themselves are unreadable there is genuinely nothing to submit, so that\n    # (and only that) is allowed to raise.\n    challenges = load_public_challenges(args.challenges)\n    _write_json(args.submission, guaranteed_fallback_submission(challenges))\n\n    base_summary = {\n        "schema": "arc-qwen38-kaggle-pass2-aggregation-v1",\n        "task_count": len(challenges),\n        "query_count": sum(len(task["test"]) for task in challenges.values()),\n    }\n    if args.guarantee_only:\n        _emit_summary({**base_summary, "status": "guaranteed-fallback"})\n        return 0\n\n    try:\n        _, receipt = aggregate(args.challenges, args.output_root, args.submission)\n        # Defensive: whatever landed on disk must be a complete, valid file.\n        validate_submission_completeness(\n            challenges, json.loads(args.submission.read_text())\n        )\n    except Exception as exc:  # never let aggregation abort the submit\n        # Fail LOUD but keep submitting: rewrite the guaranteed floor and record\n        # the failure so a degraded run is visible, not silently shipped.\n        degraded = {\n            "schema": "arc-qwen38-kaggle-aggregation-degraded-v1",\n            "observed_utc": datetime.now(timezone.utc).isoformat(),\n            "error": f"{type(exc).__name__}: {exc}",\n            "traceback": traceback.format_exc(),\n            "fallback": "guaranteed-fallback-submission-retained",\n        }\n        try:\n            _write_json(\n                args.output_root / "aggregation_degraded_receipt.json", degraded\n            )\n        except OSError:\n            pass\n        _write_json(args.submission, guaranteed_fallback_submission(challenges))\n        print(\n            "LOUD: aggregation failed; retained guaranteed-fallback submission. "\n            f"{type(exc).__name__}: {exc}",\n            file=sys.stderr,\n            flush=True,\n        )\n        _emit_summary(\n            {\n                **base_summary,\n                "status": "degraded-guaranteed-fallback",\n                "error": f"{type(exc).__name__}: {exc}",\n            }\n        )\n        return 0\n\n    _emit_summary(\n        {\n            key: receipt[key]\n            for key in (\n                "schema",\n                "observed_utc",\n                "collection_id",\n                "submission_sha256",\n                "task_count",\n                "query_count",\n                "candidate_count",\n                "tasks_with_candidate",\n                "tasks_with_visible_pass_candidate",\n                "selected_source_counts",\n                "status",\n            )\n        }\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

SANDBOX_SOURCE = '#!/usr/bin/env python3\n"""Resource-bounded Python executor for the model\'s code-interpreter tool.\n\nThis is a resource sandbox, not a security boundary: model-written code retains\nnormal filesystem and network access. Each call runs in a fresh process session\nand temporary directory with bounded address space, CPU time, wall time,\nper-file size, open files, and captured output.\nThe parent kills the call\'s original process group after the leader exits, so\nordinary forked background work is cleaned up after both success and timeout.\nDeliberately evasive code can escape these controls because this is not a\nsecurity boundary.\n\nThe file-size rlimit is per file rather than an aggregate disk quota. Combined\nwith the throwaway working directory, process-group cleanup, and wall deadline,\nit limits the practical damage from accidental writes but should not be treated\nas filesystem isolation. RLIMIT_AS is per process rather than an aggregate\ncgroup memory cap, and same-UID processes are not isolated. Defaults are sized\nwell under a 125 GB host:\n4 GiB address space, 12 CPU-s, 20 s wall, 16 MiB per file, and 8,000 retained\noutput characters.\n\nRLIMIT_NPROC is deliberately not used. On Linux it counts tasks for the whole\nUID, not descendants of this tool call; a busy notebook/vLLM user can already\nbe above a small limit and then lose even benign subprocess support. Aggregate\ntool memory is instead bounded by the notebook-wide tool-call semaphore.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nDEFAULT_LIMITS = {\n    "address_space_bytes": 4 * 1024**3,\n    "cpu_seconds": 12,\n    "wall_seconds": 20,\n    "file_size_bytes": 16 * 1024**2,\n    "max_output_chars": 8000,\n    "max_open_files": 64,\n}\n\n_INTEGER_LIMITS = {\n    "address_space_bytes",\n    "cpu_seconds",\n    "file_size_bytes",\n    "max_output_chars",\n    "max_open_files",\n}\n\n# The child bootstrap sets rlimits before loading task context. User stdout and\n# stderr are written into a bounded stream, not an unbounded StringIO later\n# sliced down. A dedicated inherited fd carries one authenticated-by-structure\n# result record; this is protocol separation, not protection from hostile code.\n_CHILD = r\'\'\'\nimport sys, json, io, os, resource, contextlib\n\nlimits = json.loads(sys.argv[1])\nresult_fd = int(sys.argv[2])\n\nrlimits = [\n    ("RLIMIT_AS", "address_space_bytes"),\n    ("RLIMIT_CPU", "cpu_seconds"),\n    ("RLIMIT_FSIZE", "file_size_bytes"),\n    ("RLIMIT_NOFILE", "max_open_files"),\n]\nfor resource_name, key in rlimits:\n    resource_id = getattr(resource, resource_name, None)\n    if resource_id is None:\n        continue\n    value = int(limits[key])\n    try:\n        resource.setrlimit(resource_id, (value, value))\n    except (ValueError, OSError):\n        pass\nif hasattr(resource, "RLIMIT_CORE"):\n    try:\n        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))\n    except (ValueError, OSError):\n        pass\n\nclass CappedTextIO(io.TextIOBase):\n    encoding = "utf-8"\n\n    def __init__(self, cap):\n        self.cap = int(cap)\n        self.parts = []\n        self.retained = 0\n        self.truncated = False\n\n    def writable(self):\n        return True\n\n    def write(self, value):\n        if not isinstance(value, str):\n            value = str(value)\n        original_length = len(value)\n        remaining = self.cap - self.retained\n        if remaining > 0:\n            kept = value[:remaining]\n            self.parts.append(kept)\n            self.retained += len(kept)\n        if original_length > max(remaining, 0):\n            self.truncated = True\n        return original_length\n\n    def flush(self):\n        return None\n\n    def getvalue(self):\n        value = "".join(self.parts)\n        if self.truncated:\n            value += "\\n...[output truncated at %d chars]" % self.cap\n        return value\n\npayload = json.load(sys.stdin)\ng = {"__name__": "__main__"}\ng.update(payload.get("context") or {})\nbuf = CappedTextIO(limits["max_output_chars"])\nresult = {"ok": True, "error": None, "stdout": ""}\ntry:\n    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):\n        exec(compile(payload["code"], "<model_code>", "exec"), g)\nexcept MemoryError:\n    result["ok"] = False\n    result["error"] = "MemoryError: hit the memory cap"\nexcept SystemExit as exc:\n    result["ok"] = False\n    result["error"] = "SystemExit(%r)" % (exc.code,)\nexcept BaseException as exc:\n    result["ok"] = False\n    result["error"] = "%s: %s" % (type(exc).__name__, str(exc)[:800])\nresult["stdout"] = buf.getvalue()\nrecord = ("RESULT:" + json.dumps(result, separators=(",", ":"))).encode()\nos.write(result_fd, record)\n\'\'\'\n\n\n@dataclass\nclass ExecResult:\n    ok: bool\n    stdout: str\n    error: str | None\n    timed_out: bool\n    killed_signal: int | None\n    wall_s: float\n\n\ndef _normalize_limits(overrides: dict | None) -> dict:\n    if overrides is not None and not isinstance(overrides, dict):\n        raise TypeError("limits must be a mapping or None")\n    unknown = sorted(set(overrides or {}) - set(DEFAULT_LIMITS))\n    if unknown:\n        raise ValueError(f"unknown resource limit(s): {unknown}")\n    limits = {**DEFAULT_LIMITS, **(overrides or {})}\n    for name in _INTEGER_LIMITS:\n        value = limits[name]\n        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:\n            raise ValueError(f"{name} must be a positive integer, got {value!r}")\n    wall = limits["wall_seconds"]\n    if isinstance(wall, bool) or not isinstance(wall, (int, float)) or wall <= 0:\n        raise ValueError(f"wall_seconds must be positive, got {wall!r}")\n    limits["wall_seconds"] = float(wall)\n    return limits\n\n\ndef _kill_process_group(process_group_id: int) -> None:\n    """Best-effort cleanup for the leader and descendants still in its session."""\n\n    try:\n        os.killpg(process_group_id, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    except PermissionError:\n        # The session is ours, but fail safely if a platform/container policy\n        # unexpectedly blocks group signaling.\n        pass\n\n\ndef _child_environment() -> dict[str, str]:\n    """Return a minimal deterministic environment with one native thread."""\n\n    environment = {\n        "PATH": os.environ.get("PATH", ""),\n        "PYTHONHASHSEED": "0",\n        "OMP_NUM_THREADS": "1",\n        "OPENBLAS_NUM_THREADS": "1",\n        "MKL_NUM_THREADS": "1",\n        "NUMEXPR_NUM_THREADS": "1",\n    }\n    # Some installations need the dynamic-loader path to import NumPy wheels.\n    if os.environ.get("LD_LIBRARY_PATH"):\n        environment["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]\n    return environment\n\n\ndef _drain_result_pipe(\n    result_fd: int,\n    *,\n    retain_bytes: int,\n    chunks: list[bytes],\n    overflow: list[bool],\n) -> None:\n    """Drain the child protocol pipe without letting it block on a full pipe.\n\n    The child emits its result only after model code finishes. Waiting for the\n    child to exit before reading can therefore deadlock when JSON escaping\n    expands retained Unicode output beyond the platform pipe capacity. Keep a\n    bounded prefix for decoding, but continue draining/discarding until EOF so\n    the child can always finish its write.\n    """\n\n    retained = 0\n    try:\n        while True:\n            chunk = os.read(result_fd, 65536)\n            if not chunk:\n                return\n            remaining = retain_bytes - retained\n            if remaining > 0:\n                kept = chunk[:remaining]\n                chunks.append(kept)\n                retained += len(kept)\n            if len(chunk) > max(remaining, 0):\n                overflow[0] = True\n    except OSError:\n        # Parent teardown may close the descriptor after process cleanup. Any\n        # incomplete record is handled by the normal "no result" path.\n        return\n\n\ndef run_python(\n    code: str,\n    context: dict | None = None,\n    limits: dict | None = None,\n) -> ExecResult:\n    """Execute ``code`` in a fresh resource-bounded child process.\n\n    ``context`` values are JSON-serialized and become globals in the child.\n    The function is designed for trusted orchestration around untrusted model\n    snippets; it bounds resources but does not prevent data or network access.\n    """\n\n    if not isinstance(code, str):\n        raise TypeError("code must be a string")\n    if context is not None and not isinstance(context, dict):\n        raise TypeError("context must be a mapping or None")\n    lim = _normalize_limits(limits)\n    payload = json.dumps(\n        {"code": code, "context": context or {}}, separators=(",", ":")\n    ).encode()\n    started = time.monotonic()\n    r_fd, w_fd = os.pipe()\n    proc = None\n    process_group_id = None\n    raw = b""\n    timed_out = False\n    killed = None\n    reader = None\n    result_chunks: list[bytes] = []\n    result_overflow = [False]\n\n    try:\n        with tempfile.TemporaryDirectory(prefix="pytool-") as cwd:\n            try:\n                proc = subprocess.Popen(\n                    [\n                        sys.executable,\n                        "-I",\n                        "-c",\n                        _CHILD,\n                        json.dumps(lim, separators=(",", ":")),\n                        str(w_fd),\n                    ],\n                    stdin=subprocess.PIPE,\n                    stdout=subprocess.DEVNULL,\n                    stderr=subprocess.DEVNULL,\n                    pass_fds=(w_fd,),\n                    cwd=cwd,\n                    start_new_session=True,\n                    env=_child_environment(),\n                )\n                process_group_id = proc.pid  # start_new_session => pgid == pid\n            finally:\n                os.close(w_fd)\n\n            # ``ensure_ascii=True`` can encode one non-BMP output character as\n            # a 12-byte surrogate-pair escape. Retain enough for the largest\n            # legitimate record and drain any excess without allocating it.\n            max_result_bytes = lim["max_output_chars"] * 12 + 65536\n            reader = threading.Thread(\n                target=_drain_result_pipe,\n                kwargs={\n                    "result_fd": r_fd,\n                    "retain_bytes": max_result_bytes,\n                    "chunks": result_chunks,\n                    "overflow": result_overflow,\n                },\n                name="safe-python-result-drain",\n                daemon=True,\n            )\n            reader.start()\n\n            try:\n                proc.communicate(input=payload, timeout=lim["wall_seconds"])\n            except subprocess.TimeoutExpired:\n                timed_out = True\n                _kill_process_group(process_group_id)\n                proc.wait()\n            finally:\n                # Also run after a normal leader exit: snippets may fork a child\n                # and return while that child keeps consuming resources.\n                if process_group_id is not None:\n                    _kill_process_group(process_group_id)\n\n            if proc.returncode and proc.returncode < 0:\n                killed = -proc.returncode\n\n            reader.join(timeout=2)\n            if reader.is_alive():\n                # All writers should be gone after process-group cleanup. Close\n                # our read descriptor as a final unblock and treat the record as\n                # incomplete rather than hanging the notebook process.\n                try:\n                    os.close(r_fd)\n                except OSError:\n                    pass\n                reader.join(timeout=1)\n            raw = b"".join(result_chunks)\n    finally:\n        try:\n            os.close(r_fd)\n        except OSError:\n            pass\n        if proc is not None and proc.poll() is None:\n            if process_group_id is not None:\n                _kill_process_group(process_group_id)\n            proc.wait()\n\n    wall = round(time.monotonic() - started, 2)\n    if timed_out:\n        return ExecResult(\n            False,\n            "",\n            f"timeout: exceeded {lim[\'wall_seconds\']:g}s wall limit",\n            True,\n            killed,\n            wall,\n        )\n\n    marker = raw.split(b"RESULT:", 1)\n    if len(marker) == 2:\n        try:\n            result = json.loads(marker[1].decode())\n            return ExecResult(\n                bool(result["ok"]),\n                result.get("stdout", ""),\n                result.get("error"),\n                False,\n                killed,\n                wall,\n            )\n        except (KeyError, TypeError, UnicodeDecodeError, json.JSONDecodeError):\n            pass\n\n    reason = f"killed by signal {killed}" if killed else "child produced no result"\n    if result_overflow[0]:\n        reason = "child result exceeded parent protocol bound"\n    if killed == signal.SIGKILL:\n        reason = "killed (CPU/memory limit)"\n    return ExecResult(False, "", reason, False, killed, wall)\n\n\n# --- OpenAI tool schema the model sees -------------------------------------\nTOOL_SCHEMA = {\n    "type": "function",\n    "function": {\n        "name": "run_python",\n        "description": (\n            "Execute Python 3 code in a fresh, resource-limited sandbox and get "\n            "its stdout back. The ARC task is preloaded as globals: `train` (a "\n            "list of {\\"input\\": grid, \\"output\\": grid} pairs) and `test_input` "\n            "(the grid to solve); grids are lists of lists of ints 0-9. Import "\n            "anything you need (e.g. numpy). PRINT what you want to inspect -- only "\n            "stdout is returned. State does NOT persist between calls; include all "\n            "needed code each time. Limits: ~4 GiB RAM, ~12 CPU-s, 20 s wall. "\n            "This is resource isolation, not filesystem/network isolation."\n        ),\n        "parameters": {\n            "type": "object",\n            "properties": {\n                "code": {\n                    "type": "string",\n                    "description": "Python 3 source to run.",\n                }\n            },\n            "required": ["code"],\n            "additionalProperties": False,\n        },\n    },\n}\n\n\ndef _selftest() -> int:\n    cases = [\n        (\n            "benign",\n            "print(\'rows:\', len(test_input)); print(sum(sum(r) for r in test_input))",\n            {"test_input": [[1, 2], [3, 4]]},\n            {},\n            lambda result: result.ok and "10" in result.stdout,\n        ),\n        (\n            "memory_bomb",\n            "x = [0] * (10**11)",\n            {},\n            {},\n            lambda result: not result.ok,\n        ),\n        (\n            "infinite_loop",\n            "while True:\\n    pass",\n            {},\n            {},\n            lambda result: not result.ok,\n        ),\n        (\n            "output_bomb",\n            "print(\'x\' * (10**8))",\n            {},\n            {},\n            lambda result: result.ok and "[output truncated" in result.stdout,\n        ),\n        (\n            "unicode_pipe",\n            "print(\'😀\' * 6000)",\n            {},\n            {"wall_seconds": 3, "cpu_seconds": 2},\n            lambda result: result.ok\n            and not result.timed_out\n            and result.stdout.count("😀") == 6000,\n        ),\n        (\n            "numpy",\n            "import numpy as np; print(np.array(test_input).T.tolist())",\n            {"test_input": [[1, 2], [3, 4]]},\n            {},\n            lambda result: result.ok and "[[1, 3], [2, 4]]" in result.stdout,\n        ),\n        (\n            "user_error",\n            "raise ValueError(\'boom\')",\n            {},\n            {},\n            lambda result: not result.ok and "ValueError" in (result.error or ""),\n        ),\n    ]\n    failures = 0\n    for name, source, context, limit_overrides, predicate in cases:\n        result = run_python(\n            source,\n            context,\n            limits={\n                "wall_seconds": 8,\n                "cpu_seconds": 5,\n                **limit_overrides,\n            },\n        )\n        passed = bool(predicate(result))\n        failures += not passed\n        preview = (result.stdout or result.error or "").replace("\\n", " ")[:80]\n        print(\n            f"{name:14s} pass={passed!s:5s} ok={result.ok!s:5s} "\n            f"timed_out={result.timed_out!s:5s} "\n            f"wall={result.wall_s:5.2f}s -> {preview}"\n        )\n    if failures:\n        print(f"SELFTEST FAILED: {failures} case(s)", file=sys.stderr)\n        return 1\n    print("SELFTEST PASS")\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(_selftest())\n'

def load_module_from_source(name, path, source):
    """Write and import an embedded module with normal import semantics."""

    path.write_text(source)
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"could not create a loader for {name!r} at {path}")
    module = importlib.util.module_from_spec(spec)
    previous = sys.modules.get(name)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except BaseException:
        if previous is None:
            sys.modules.pop(name, None)
        else:
            sys.modules[name] = previous
        raise
    return module

def stable_sample_seed(task_id, query_index, slot):
    """Derive a deterministic, task-specific 31-bit sampling seed."""

    payload = f"{task_id}:{query_index}:{slot}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") & 0x7FFFFFFF

def select_distinct_fallback(model_grid, fallback_row):
    """Choose a prevalidated fallback distinct from a first model hypothesis."""

    for name in ("attempt_2", "attempt_1"):
        candidate = fallback_row[name]
        if candidate != model_grid:
            return candidate
    raise ValueError("fallback pair unexpectedly contains no grid distinct from model output")

def parse_model_grid(text, *, require_closed_fence=False):
    """Return the last rectangular digit grid, preferring closed fences.

    A response stopped at a token limit can contain a rectangular prefix of an
    unfinished grid. Callers can require a closed fence in that case.
    """

    if not isinstance(text, str):
        return None

    def parse_block(block):
        rows = []
        for line in (block or "").strip().splitlines():
            tokens = line.strip().split()
            if tokens and all(token.isdigit() and len(token) == 1 for token in tokens):
                rows.append([int(token) for token in tokens])
            elif rows:
                break
        if not rows or any(len(row) != len(rows[0]) for row in rows):
            return None
        return rows

    fenced = re.findall(r"```(?:grid|text)?\s*\n(.*?)```", text or "", re.DOTALL)
    for candidate in reversed(fenced):
        grid = parse_block(candidate)
        if grid is not None:
            return grid

    if require_closed_fence:
        return None

    blocks = []
    current = []
    for line in (text or "").splitlines() + [""]:
        tokens = line.strip().split()
        if tokens and all(token.isdigit() and len(token) == 1 for token in tokens):
            current.append(line)
        elif current:
            blocks.append("\n".join(current))
            current = []
    for candidate in reversed(blocks):
        grid = parse_block(candidate)
        if grid is not None:
            return grid
    return None

def response_reasoning(message):
    """Read vLLM's current field while retaining older-server compatibility."""

    if not isinstance(message, dict):
        return ""
    value = message.get("reasoning") or message.get("reasoning_content") or ""
    return value if isinstance(value, str) else ""

def assistant_replay_message(message):
    """Normalize one assistant tool-call turn for vLLM history replay."""

    if not isinstance(message, dict):
        raise TypeError("assistant replay message must be an object")
    content = message.get("content") or ""
    if not isinstance(content, str):
        content = ""
    tool_calls = message.get("tool_calls")
    if tool_calls is None:
        tool_calls = []
    elif not isinstance(tool_calls, list):
        raise TypeError("assistant replay tool_calls must be a list")
    replay = {
        "role": "assistant",
        "content": content,
        "tool_calls": tool_calls,
    }
    reasoning = response_reasoning(message)
    if reasoning:
        replay["reasoning"] = reasoning
    return replay

def decode_tool_code(function):
    """Accept OpenAI JSON-string arguments and already-decoded mappings."""

    if not isinstance(function, dict):
        return ""
    arguments = function.get("arguments") or {}
    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except (TypeError, json.JSONDecodeError):
            return ""
    if not isinstance(arguments, dict):
        return ""
    code = arguments.get("code", "")
    return code if isinstance(code, str) else ""

def tool_wall_budget_seconds(deadline_s, now_s, default_wall_s=20.0):
    """Return a tool wall limit that cannot outlive its episode deadline."""

    default_wall_s = float(default_wall_s)
    if default_wall_s <= 0:
        raise ValueError("default tool wall limit must be positive")
    remaining_s = float(deadline_s) - float(now_s)
    if remaining_s <= 0:
        return None
    return min(default_wall_s, remaining_s)

def render(g): return "\n".join(" ".join(str(v) for v in row) for row in g)

def build_agent_prompt(task):
    parts = ["You are solving an abstract reasoning puzzle. Each training example maps "
             "an INPUT grid to an OUTPUT grid (digits 0-9 are colors; 0 is usually "
             "background). Infer the ONE rule that maps every input to its output, "
             "then apply it to the FINAL INPUT.\n",
             "You have a Python tool `run_python`. The task is preloaded there as "
             "`train` (a list of {\"input\": grid, \"output\": grid}) and `test_input` "
             "(the grid to solve). Use it however you find helpful -- to inspect grids, "
             "test a transformation against every example, or compute the answer -- or "
             "just reason directly. It is entirely your choice.\n"]
    for i, p in enumerate(task["train"]):
        parts.append("Example %d INPUT:\n%s\nExample %d OUTPUT:\n%s\n" %
                     (i+1, render(p["input"]), i+1, render(p["output"])))
    parts.append("FINAL INPUT:\n%s\n" % render(task["test_input"]))
    parts.append("When you are confident, give the final output grid as rows of "
                 "space-separated digits inside a fenced block:\n```grid\n<rows>\n```")
    return "\n".join(parts)

def build_prompt(task):
    parts = ["You are solving an abstract reasoning puzzle. Each example has an "
             "INPUT grid and its OUTPUT grid (digits 0-9 are colors; 0 is usually "
             "background). Infer the ONE rule that maps every input to its output, "
             "then apply it to the FINAL INPUT.\n"]
    for i, p in enumerate(task["train"]):
        parts.append("Example %d INPUT:\n%s\nExample %d OUTPUT:\n%s\n" %
                     (i+1, render(p["input"]), i+1, render(p["output"])))
    parts.append("FINAL INPUT:\n%s\n" % render(task["test_input"]))
    parts.append("Reason step by step however you like. Then give ONLY the final "
                 "output grid as rows of space-separated digits inside a fenced "
                 "block:\n```grid\n<rows>\n```\nOutput nothing after the block.")
    return "\n".join(parts)


In [ ]:
%%writefile /kaggle/working/ab/a100/serve.sh
#!/usr/bin/env bash
# Start one vLLM 0.19 server per group of TP GPUs for a100/qwen_ab.py (docs/A100_PLAN.md, steps 3-4).
#
#   MODEL=/data/qwen3-8-27b-fp8 VENV=/tmp/vllm_env TP=1 SEQS=8 bash a100/serve.sh
#
# GPUs   the ones in CUDA_VISIBLE_DEVICES (as assigned by the scheduler), else all GPUs nvidia-smi lists
# TP     tensor parallel size per server (1 on 80GB, 2 on 40GB); the GPU count must be a multiple of it
# SEQS   max concurrent sequences per server (= harness --concurrency / servers)
# MTP=1  add MTP speculative decoding (start without it first, per the plan)
# KVFP8=1  fp8 KV cache with the Triton backend (the L4 production setting; off on A100 by default)
# TOOLS=0  start without the tool-call parser (only for the tools-off condition)
# VENV   optional pip --target dir holding vLLM (the Kaggle cp311 wheelhouse install); launched with python -S
# Writes $LOGDIR/servers.json (pass it to qwen_ab.py --servers) and $LOGDIR/vllm_<port>.log. Default LOGDIR=server_logs.
set -euo pipefail
: "${MODEL:?set MODEL to the Qwen3.8-27B-FP8 directory}"
TP=${TP:-1}; SEQS=${SEQS:-8}; CTX=${CTX:-131072}; MEM=${MEM:-0.92}; PORT0=${PORT0:-8000}
NAME=${NAME:-Qwen/Qwen3.8-27B-FP8}; LOGDIR=${LOGDIR:-server_logs}; PY=${PY:-python}
mkdir -p "$LOGDIR"

if [ -n "${CUDA_VISIBLE_DEVICES:-}" ]; then IFS=, read -ra gpus <<< "$CUDA_VISIBLE_DEVICES"
else mapfile -t gpus < <(nvidia-smi --query-gpu=index --format=csv,noheader); fi
((${#gpus[@]} > 0 && ${#gpus[@]} % TP == 0)) || { echo "GPUs (${gpus[*]}) not a positive multiple of TP=$TP"; exit 1; }
nserv=$((${#gpus[@]} / TP))

flags=(--served-model-name "$NAME" --tensor-parallel-size "$TP" --max-model-len "$CTX" --gpu-memory-utilization "$MEM"
       --enable-prefix-caching --max-num-seqs "$SEQS" --reasoning-parser qwen3 --language-model-only --dtype bfloat16)
[ "${TOOLS:-1}" = 1 ] && flags+=(--enable-auto-tool-choice --tool-call-parser qwen3_coder)
[ "${MTP:-0}" = 1 ] && flags+=(--speculative-config '{"method":"mtp","num_speculative_tokens":3}')
if [ "${KVFP8:-0}" = 1 ]; then flags+=(--kv-cache-dtype fp8 --attention-backend TRITON_ATTN); export VLLM_FORCE_ATTN_BACKEND=TRITON_ATTN; fi
export VLLM_USE_FLASHINFER_SAMPLER=0
export USE_TF=0 TRANSFORMERS_NO_TF=1   # Kaggle images ship TensorFlow; transformers importing it crashes vLLM's model inspection
pyargs=(); if [ -n "${VENV:-}" ]; then pyargs=(-S); export PYTHONPATH="$VENV" PATH="$VENV/bin:$PATH"; fi
up() { python3 -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:$1/v1/models', timeout=5)" 2> /dev/null; }

pids=(); devs_list=()
for ((s = 0; s < nserv; s++)); do
  port=$((PORT0 + s)); devs=$(IFS=,; echo "${gpus[*]:$((s * TP)):$TP}")
  if up "$port"; then echo "port $port already serves something; stop it first"; exit 1; fi
  echo "server $s: GPUs $devs port $port flags: ${flags[*]}"
  CUDA_VISIBLE_DEVICES=$devs nohup "$PY" "${pyargs[@]}" -m vllm.entrypoints.openai.api_server --model "$MODEL" \
    --port "$port" "${flags[@]}" > "$LOGDIR/vllm_$port.log" 2>&1 &
  pids+=($!); devs_list+=("$devs")
done
for ((s = 0; s < nserv; s++)); do
  port=$((PORT0 + s)); t0=$SECONDS
  until up "$port"; do
    sleep 5
    if ! kill -0 "${pids[$s]}" 2> /dev/null; then echo "server on port $port died"; tail -30 "$LOGDIR/vllm_$port.log"; exit 1; fi
    if ((SECONDS - t0 > 3600)); then echo "port $port not ready after 1h; see $LOGDIR/vllm_$port.log"; exit 1; fi
  done
  kill -0 "${pids[$s]}" || { echo "server on port $port exited after answering"; exit 1; }
  echo "ready: port $port (pid ${pids[$s]}) after $((SECONDS - t0))s"
  grep -iE "quantiz|marlin|kv cache|maximum concurrency" "$LOGDIR/vllm_$port.log" | tail -5 || true
done

vllm_version=$("$PY" "${pyargs[@]}" -c "import vllm; print(vllm.__version__)" 2> /dev/null || echo unknown)
gpu_names=$(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | sort -u | paste -sd ";")
python3 - "$LOGDIR/servers.json" "$MODEL" "$vllm_version" "$gpu_names" "$PORT0" "${devs_list[@]}" -- "${flags[@]}" <<'EOF'
import json, sys
out, model, version, gpus, port0, *rest = sys.argv[1:]
cut = rest.index("--")
devs, flags = rest[:cut], rest[cut + 1:]
json.dump({"model_path": model, "vllm": version, "gpus": gpus, "flags": flags,
           "servers": [{"url": f"http://127.0.0.1:{int(port0) + s}", "gpus": d} for s, d in enumerate(devs)]},
          open(out, "w"), indent=1)
print("wrote", out)
EOF


In [ ]:
%%writefile /kaggle/working/ab/a100/qwen_v7.py
"""v7 Qwen runner: one episode per TASK, with a train-verified program as the main way to answer.

What changes against a100/qwen_ab.py (the hybrid6 harness), following what the top public reports describe
(discussion 748082 / 750546: "a proper iterative feedback harness", "python repl should do almost all the thinking"):
  - the model is asked for `transform(grid)`. Every run_python call that defines it is followed by an automatic
    check on all training pairs (a second sandbox call); the per-pair result (OK / shape / differing cells) is appended
    to the tool output, so the model gets exact feedback without spending tokens on writing the check itself
  - when `transform` matches every training pair, it is applied to every test input and the episode ENDS there
    (finish = "verified"). In hybrid6, 20 of 32 confirm samples ran out of the 64k budget; most tokens went into
    the model re-deriving and re-checking by hand
  - one episode per task (all test inputs at once) instead of one per output
  - a direct answer is still accepted as before: natural stop with one ```grid block per test input
  - slot 1 (second sample) is launched only for tasks that have no verified answer yet

Same budget rules as qwen_ab.py: a TOTAL completion-token budget over all turns, prompt counted by the server's
/tokenize, no forced final turn. Writes samples.jsonl in the qwen_ab.py row format (one row per test output, with
`verified`), so the merge stays per output.

    python a100/qwen_v7.py --cohort live --effort medium --budget 65536 --samples 2 --out runs/v7 \
        --servers server_logs/servers.json --fixed ab/live --challenges ab/live/challenges.json
"""
import argparse
import concurrent.futures as cf
import copy
import datetime
import glob
import json
import os
import pathlib
import re
import sys
import threading
import time
import urllib.error
import urllib.request
import uuid

HERE = pathlib.Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import runner_parts as rp  # noqa: E402

MODEL_NAME = rp.RUNTIME_CONFIG["model"]["served_name"]
SETTINGS = rp.RUNTIME_CONFIG["sampling"]

ap = argparse.ArgumentParser()
ap.add_argument("--cohort", required=True, help="live | explore | confirm | tid,tid,...")
ap.add_argument("--effort", required=True, choices=["low", "medium", "high", "xhigh"])
ap.add_argument("--budget", type=int, required=True, help="total completion tokens per task episode, all turns")
ap.add_argument("--samples", type=int, default=1, help="episodes per task; slot>0 only for tasks not yet verified")
ap.add_argument("--max-turns", type=int, default=64)
ap.add_argument("--context-len", type=int, default=131072)
ap.add_argument("--servers", required=True)
ap.add_argument("--concurrency", type=int, default=16)
ap.add_argument("--tool-concurrency", type=int, default=8)
ap.add_argument("--fixed", required=True, help="dir with qwen_eligible_<cohort>.json (or _live), nvarc_*.json")
ap.add_argument("--challenges", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--limit", type=int, default=0)
ap.add_argument("--stop-launch-at", type=float, default=0)
ap.add_argument("--dry-check", action="store_true", help="only run the sandbox self-tests (no server)")
args = ap.parse_args()

# ---- the tool file guard of qwen_ab.py (the answer files are attached on Kaggle) ------------------------------------
FILE_GUARD = '''import tempfile as _tempfile
os.chdir(_tempfile.mkdtemp())
def _deny_task_files(event, args, _events=frozenset({"open", "os.listdir", "os.scandir", "glob.glob", "os.chdir",
        "os.system", "subprocess.Popen", "os.exec", "os.posix_spawn", "os.spawn", "shutil.copyfile", "os.symlink",
        "os.link", "os.rename"}), _bad=("kaggle", "solution", "input")):
    if event in _events and any(b in repr(args).lower() for b in _bad):
        raise PermissionError("file access outside the task data is disabled")
sys.addaudithook(_deny_task_files)
'''
GUARD_AT = "payload = json.load(sys.stdin)\n"
assert rp.SANDBOX_SOURCE.count(GUARD_AT) == 1
TMP = pathlib.Path(os.environ.get("TMPDIR", "/tmp"))
SANDBOX = rp.load_module_from_source("qwen38_v7_python_tool", TMP / "qwen38_v7_python_tool.py",
                                     rp.SANDBOX_SOURCE.replace(GUARD_AT, GUARD_AT + FILE_GUARD))
AGG = rp.load_module_from_source("qwen38_v7_aggregator", TMP / "qwen38_v7_aggregator.py", rp.AGGREGATOR_SOURCE)

TOOL_SCHEMA = copy.deepcopy(SANDBOX.TOOL_SCHEMA)
TOOL_SCHEMA["function"]["description"] = (
    "Execute Python 3 code in a fresh, resource-limited sandbox and get its stdout back. Preloaded globals: `train` "
    "(a list of {\"input\": grid, \"output\": grid} pairs), `test_inputs` (list of the test input grids) and "
    "`test_input` (= test_inputs[0]); grids are lists of lists of ints 0-9. numpy is available. PRINT what you want "
    "to see -- only stdout is returned. State does NOT persist between calls; include all needed code each time. "
    "If your code defines `transform(grid)`, it is then automatically checked on every training pair and the result "
    "is appended to the output. Limits: ~4 GiB RAM, ~12 CPU-s, 20 s wall.")

# Runs in its own sandbox call after the model's code: re-executes that code with its output discarded, then checks
# `transform` on deep copies of the training inputs. Prints one marker line the runner parses (and strips).
CHECK_SOURCE = r'''
import contextlib as _cl, copy as _cp, io as _io, json as _json
def _as_grid(x):
    if hasattr(x, "tolist"):
        x = x.tolist()
    return [[int(v) for v in row] for row in x]
_ns = {"__name__": "__main__", "train": _cp.deepcopy(train), "test_inputs": _cp.deepcopy(test_inputs),
       "test_input": _cp.deepcopy(test_inputs[0])}
_res = {"defined": False, "train": [], "test": None}
try:
    with _cl.redirect_stdout(_io.StringIO()), _cl.redirect_stderr(_io.StringIO()):
        exec(compile(MODEL_CODE, "<model_code>", "exec"), _ns)
    _f = _ns.get("transform")
    _res["defined"] = callable(_f)
except BaseException as _e:
    _res["load_error"] = "%s: %s" % (type(_e).__name__, str(_e)[:200])
    _f = None
if callable(_f):
    for _ex in train:
        try:
            with _cl.redirect_stdout(_io.StringIO()), _cl.redirect_stderr(_io.StringIO()):
                _out = _as_grid(_f(_cp.deepcopy(_ex["input"])))
            _want = _ex["output"]
            if _out == _want:
                _res["train"].append({"ok": True})
            elif len(_out) != len(_want) or any(len(r) != len(_want[0]) for r in _out):
                _res["train"].append({"ok": False, "shape": [len(_out), len(_out[0]) if _out else 0],
                                      "want_shape": [len(_want), len(_want[0])]})
            else:
                _bad = [(r, c, _out[r][c], _want[r][c]) for r in range(len(_want)) for c in range(len(_want[0]))
                        if _out[r][c] != _want[r][c]]
                _res["train"].append({"ok": False, "n_diff": len(_bad), "cells": len(_want) * len(_want[0]),
                                      "first": _bad[:6]})
        except BaseException as _e:
            _res["train"].append({"ok": False, "error": "%s: %s" % (type(_e).__name__, str(_e)[:160])})
    if _res["train"] and all(_r["ok"] for _r in _res["train"]):
        _res["test"] = []
        for _ti in test_inputs:
            try:
                with _cl.redirect_stdout(_io.StringIO()), _cl.redirect_stderr(_io.StringIO()):
                    _res["test"].append(_as_grid(_f(_cp.deepcopy(_ti))))
            except BaseException as _e:
                _res["test"].append("%s: %s" % (type(_e).__name__, str(_e)[:160]))
print("\n__V7CHECK__" + _json.dumps(_res))
'''
CHECK_MARK = "__V7CHECK__"


def valid_grid(grid):
    if grid is None:
        return None
    try:
        AGG.validate_grid(grid, context="v7 grid")
        return grid
    except Exception:
        return None


def build_prompt(task):
    tests = [t["input"] for t in task["test"]]
    parts = ["You are solving an abstract reasoning puzzle. Each training example maps an INPUT grid to an OUTPUT grid "
             "(digits 0-9 are colors; 0 is usually background). Infer the ONE rule that maps every input to its output, "
             f"then apply it to the test input{'s' if len(tests) > 1 else ''}.", "",
             "You have a Python tool `run_python`. Preloaded there: `train` (list of {\"input\": grid, \"output\": grid}), "
             "`test_inputs` (list of the test input grids) and `test_input` (= test_inputs[0]).", "",
             "Recommended way to answer: write a Python function `transform(grid)` (list of lists of ints in, list of "
             "lists out) that implements the rule. Every time your run_python code defines `transform`, it is "
             "automatically run on ALL training pairs and you get back which ones match and where they differ. As soon "
             "as it matches every training pair, it is applied to the test input(s) and that is your final answer: "
             "you are done. So test hypotheses with code early, and make `transform` general (do not hard-code "
             "training outputs).", ""]
    for n, ex in enumerate(task["train"], 1):
        parts += [f"Example {n} INPUT:", rp.render(ex["input"]), f"Example {n} OUTPUT:", rp.render(ex["output"]), ""]
    for n, g in enumerate(tests, 1):
        parts += [f"TEST INPUT {n}:" if len(tests) > 1 else "TEST INPUT:", rp.render(g), ""]
    parts += ["If you cannot get a program to match, you may answer directly instead: give the output grid as rows of "
              "space-separated digits inside a fenced block" +
              (f", one block per test input in order ({len(tests)} blocks):" if len(tests) > 1 else ":"),
              "```grid", "<rows>", "```"]
    return "\n".join(parts)


def parse_grids(text, n):
    """n == 1: the qwen_ab.py rule (last closed grid). n > 1: the last n closed ```grid blocks, in order."""
    if n == 1:
        g = valid_grid(rp.parse_model_grid(text, require_closed_fence=True))
        return [g] if g else None
    blocks = re.findall(r"```(?:grid|text)?\s*\n(.*?)```", text or "", re.DOTALL)
    grids = [valid_grid(rp.parse_model_grid("```grid\n" + b + "```", require_closed_fence=True)) for b in blocks]
    grids = [g for g in grids if g]
    return grids[-n:] if len(grids) >= n else None


def summarize_check(res, task):
    """Feedback text for the model, and the verified test outputs (or None)."""
    if not res.get("defined"):
        return ("[auto-check] `transform` could not be loaded: " + res["load_error"]) if res.get("load_error") else "", None
    lines, n_ok = [], 0
    for k, r in enumerate(res["train"], 1):
        if r["ok"]:
            n_ok += 1
            lines.append(f"train {k}: OK")
        elif "error" in r:
            lines.append(f"train {k}: raised {r['error']}")
        elif "shape" in r:
            lines.append(f"train {k}: wrong shape {r['shape'][0]}x{r['shape'][1]} (expected "
                         f"{r['want_shape'][0]}x{r['want_shape'][1]})")
        else:
            cells = ", ".join(f"({a},{b}) got {g} want {w}" for a, b, g, w in r["first"])
            lines.append(f"train {k}: {r['n_diff']} of {r['cells']} cells differ, e.g. {cells}")
    head = f"[auto-check of transform] {n_ok}/{len(res['train'])} training pairs match"
    if res.get("test") is None:
        return head + "\n" + "\n".join(lines), None
    outs = [valid_grid(g) if isinstance(g, list) else None for g in res["test"]]
    if any(g is None for g in outs):
        bad = [str(g)[:160] for g in res["test"] if not isinstance(g, list) or valid_grid(g) is None]
        return head + ", but it fails on a test input: " + "; ".join(bad) + "\nMake it work on the test input(s).", None
    copies = {json.dumps(ex["output"]) for ex in task["train"]} | {json.dumps(t["input"]) for t in task["test"]}
    if any(json.dumps(g) in copies for g in outs):
        return head + ", but a test output is an exact copy of a training output or of the test input, which means " \
                      "the rule is hard-coded or not applied. Make `transform` general.", None
    return head + ". Verified.", outs


OUT = pathlib.Path(args.out)
(OUT / "raw").mkdir(parents=True, exist_ok=True)
challenges = json.load(open(args.challenges))
fixed = pathlib.Path(args.fixed)
if args.cohort in ("live", "explore", "confirm"):
    eligible = [tuple(x) for x in json.load(open(fixed / f"qwen_eligible_{args.cohort}.json"))]
    tasks = list(dict.fromkeys(t for t, _ in eligible))      # task order = first eligible output (priority order)
else:
    tasks = args.cohort.split(",")
TOOL_SLOTS = threading.BoundedSemaphore(args.tool_concurrency)
WRITE_LOCK = threading.Lock()
VERIFIED = set()


def run_tool(code, task):
    tests = [t["input"] for t in task["test"]]
    ctx = {"train": task["train"], "test_inputs": tests, "test_input": tests[0]}
    with TOOL_SLOTS:
        res = SANDBOX.run_python(code, ctx)
    out = res.stdout if res.ok else "ERROR: " + (res.error or "")
    wall = res.wall_s or 0
    check, verified = None, None
    if "transform" in code and re.search(r"def\s+transform\s*\(|transform\s*=", code):
        with TOOL_SLOTS:
            cres = SANDBOX.run_python(CHECK_SOURCE, {**ctx, "MODEL_CODE": code})
        wall += cres.wall_s or 0
        mark = (cres.stdout or "").rfind(CHECK_MARK)
        if cres.ok and mark >= 0:
            try:
                check = json.loads(cres.stdout[mark + len(CHECK_MARK):].strip().splitlines()[0])
            except Exception:
                check = None
        if check is None:
            fb = "[auto-check] could not run transform: " + (cres.error or "timeout or no result")[:300]
        else:
            fb, verified = summarize_check(check, task)
        if fb:
            out = (out or "") + "\n" + fb
    return res.ok, out or "(no output)", wall, check, verified


def selftest():
    task = {"train": [{"input": [[1, 2]], "output": [[2, 1]]}, {"input": [[3, 4, 5]], "output": [[5, 4, 3]]}],
            "test": [{"input": [[7, 8]]}, {"input": [[1, 2, 3]]}]}
    ok, out, _, _, ver = run_tool("def transform(g):\n    return [r[::-1] for r in g]\nprint('hi')", task)
    assert ok and "hi" in out and ver == [[[8, 7]], [[3, 2, 1]]] and "2/2" in out, out
    ok, out, _, _, ver = run_tool("import numpy as np\ndef transform(g):\n    return np.array(g)", task)
    assert ver is None and "0/2" in out and "cells differ" in out, out
    ok, out, _, _, ver = run_tool("def transform(g):\n    return [[2, 1]] if g == [[1, 2]] else [[5, 4, 3]]", task)
    assert ver is None and "copy" in out, out          # lookup table: test output equals a training output
    ok, out, _, _, ver = run_tool("def transform(g):\n    raise ValueError('x')", task)
    assert ver is None and "raised ValueError" in out, out
    ok, out, _, _, ver = run_tool("x = 1/0", task)
    assert not ok and ver is None, out
    visible = glob.glob("/kaggle/input/**/*solution*.json", recursive=True)[:1]
    for probe in [f"print(open({v!r}).read()[:40])" for v in visible] + ["import os; print(os.listdir('/kaggle/input'))"]:
        ok, out, _, _, _ = run_tool(probe, task)
        assert not (ok and ("{" in out or "solution" in out.lower())), f"sandbox leaked: {out[:200]}"
    print("v7 tool self-test ok", flush=True)


selftest()
if args.dry_check:
    sys.exit(0)

URLS = [s["url"] for s in json.load(open(args.servers))["servers"]]


def post(url, path, body, timeout):
    req = urllib.request.Request(url + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())


def chat_kwargs():
    return {"enable_thinking": SETTINGS["enable_thinking"], "reasoning_effort": args.effort}


def count_prompt(url, messages):
    body = {"model": MODEL_NAME, "messages": messages, "add_generation_prompt": True, "chat_template_kwargs": chat_kwargs(),
            "tools": [TOOL_SCHEMA]}
    return int(post(url, "/tokenize", body, 120)["count"])


def append(name, row):
    with WRITE_LOCK, open(OUT / name, "a") as f:
        f.write(json.dumps(row) + "\n")


class InfraError(Exception):
    pass


class NotStarted(Exception):
    pass


def solve(tid, slot, url):
    task = challenges[tid]
    n_test = len(task["test"])
    messages = [{"role": "user", "content": build_prompt(task)}]
    seed = rp.stable_sample_seed(tid, -1, slot)
    key, attempt = f"{tid}_{slot}", uuid.uuid4().hex[:8]
    used = turns = tool_calls = tool_errors = checks = 0
    best_train, tool_s = -1, 0.0
    finish, grids, raw = "max_turns", None, []
    t0 = time.time()
    for turn in range(args.max_turns):
        try:
            prompt_tokens = count_prompt(url, messages)
        except Exception as exc:
            raise InfraError(f"tokenize: {type(exc).__name__}: {str(exc)[:200]}") from exc
        if args.budget - used <= 0:
            finish = "budget_exhausted"
            break
        max_tokens = min(args.budget - used, args.context_len - prompt_tokens)
        if max_tokens <= 0:
            finish = "context_exhausted"
            break
        body = {"model": MODEL_NAME, "messages": messages, "temperature": SETTINGS["temperature"],
                "top_p": SETTINGS["top_p"], "top_k": SETTINGS["top_k"], "max_tokens": max_tokens, "seed": seed,
                "chat_template_kwargs": chat_kwargs(), "tools": [TOOL_SCHEMA], "tool_choice": "auto"}
        ts = time.time()
        try:
            p = post(url, "/v1/chat/completions", body, 4 * 3600)
        except urllib.error.HTTPError as exc:
            raise InfraError("http%d: %s" % (exc.code, exc.read().decode(errors="replace")[:300])) from exc
        except Exception as exc:
            raise InfraError(f"chat: {type(exc).__name__}: {str(exc)[:200]}") from exc
        turns += 1
        ch = p["choices"][0]
        msg = ch["message"]
        usage = p.get("usage") or {}
        n_out = usage.get("completion_tokens") or 0
        used += n_out
        content = msg.get("content") if isinstance(msg.get("content"), str) else ""
        reasoning = rp.response_reasoning(msg)
        tcs = msg.get("tool_calls") or []
        turn_finish = ch.get("finish_reason")
        append("turns.jsonl", {"key": key, "attempt": attempt, "turn": turn, "prompt_tokens": prompt_tokens,
                               "max_tokens": max_tokens, "completion_tokens": n_out, "finish_reason": turn_finish,
                               "tool_calls": len(tcs), "reasoning_chars": len(reasoning), "content_chars": len(content),
                               "wall_s": round(time.time() - ts, 1)})
        raw.append({"turn": turn, "finish_reason": turn_finish, "message": msg, "usage": usage})
        if not tcs and ("<tool_call>" in reasoning or "<tool_call>" in content):
            finish = "protocol_error:tool_call_markup"
            break
        if tcs and turn_finish != "length":
            messages.append(rp.assistant_replay_message(msg))
            for tc in tcs:
                fn = tc.get("function") or {}
                code = rp.decode_tool_code(fn) if fn.get("name") == "run_python" else ""
                tool_calls += 1
                if not code:
                    out, ok, check, verified = ("ERROR: run_python arguments did not contain non-empty code"
                                                if fn.get("name") == "run_python" else "error: unknown tool %r" % fn.get("name")), \
                        False, None, None
                else:
                    ok, out, wall, check, verified = run_tool(code, task)
                    tool_s += wall
                tool_errors += 0 if ok else 1
                if check is not None:
                    checks += 1
                    best_train = max(best_train, sum(r["ok"] for r in check.get("train", [])))
                messages.append({"role": "tool", "tool_call_id": tc.get("id"), "name": fn.get("name") or "unknown_tool",
                                 "content": out})
                if verified:
                    grids, finish = verified, "verified"
                    break
            if finish == "verified":
                break
            continue
        finish = turn_finish or "stop"
        grids = parse_grids(content, n_test) if finish == "stop" else None
        break
    row = {"task_id": tid, "slot": slot, "seed": seed, "attempt": attempt, "finish": finish,
           "verified": finish == "verified", "grids": grids, "n_test": n_test, "completion_tokens": used,
           "turns": turns, "tool_calls": tool_calls, "tool_errors": tool_errors, "checks": checks,
           "best_train_ok": best_train, "n_train": len(task["train"]), "tool_s": round(tool_s, 1),
           "wall_s": round(time.time() - t0, 1), "server": url}
    (OUT / "raw" / f"{key}.json").write_text(json.dumps({"messages": messages, "responses": raw, "result": row}))
    with WRITE_LOCK:
        if row["verified"]:
            VERIFIED.add(tid)
        with open(OUT / "tasks.jsonl", "a") as f:
            f.write(json.dumps(row) + "\n")
        with open(OUT / "samples.jsonl", "a") as f:   # qwen_ab.py row format, one row per test output
            for qi in range(n_test):
                f.write(json.dumps({"task_id": tid, "query_index": qi, "slot": slot, "finish": finish,
                                    "verified": row["verified"], "grid": grids[qi] if grids else None,
                                    "completion_tokens": used, "turns": turns}) + "\n")
    return row


def run_one(t, s, url):
    if args.stop_launch_at and time.time() > args.stop_launch_at:
        raise NotStarted()
    if s > 0 and t in VERIFIED:
        return None
    try:
        return solve(t, s, url)
    except Exception as exc:
        append("failures.jsonl", {"task_id": t, "slot": s, "server": url,
                                  "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                                  "error": f"{type(exc).__name__}: {str(exc)[:400]}"})
        raise


done = {}
if (OUT / "tasks.jsonl").exists():
    for line in open(OUT / "tasks.jsonl"):
        r = json.loads(line)
        done[(r["task_id"], r["slot"])] = r
        if r["verified"]:
            VERIFIED.add(r["task_id"])


def next_slot(t):
    """First slot of task t still to run, or None (verified, or all slots done)."""
    for s in range(args.samples):
        if t in VERIFIED:
            return None
        if (t, s) not in done:
            return s
    return None


# Slot 0 of every task first (priority order); slot s+1 of a task is queued only when its slot s ended unverified,
# so it goes behind everything already queued and never runs alongside the same task's earlier slot.
work = [(t, next_slot(t)) for t in tasks if next_slot(t) is not None]
if args.limit:
    work = work[:args.limit]
(OUT / "manifest.json").write_text(json.dumps({"args": vars(args), "tasks": tasks, "urls": URLS,
                                               "utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}, indent=1))
print(f"{len(tasks)} tasks x {args.samples} slots; {len(done)} done; starting {len(work)}", flush=True)
t_start = time.time()
tokens, failed, skipped, n = [0], [0], [0], 0
with cf.ThreadPoolExecutor(args.concurrency) as ex:
    futs = {ex.submit(run_one, t, s, URLS[k % len(URLS)]): (t, s) for k, (t, s) in enumerate(work)}
    while futs:
        fin, _ = cf.wait(futs, return_when=cf.FIRST_COMPLETED)
        for fut in fin:
            t, s = futs.pop(fut)
            n += 1
            try:
                r = fut.result()
            except NotStarted:
                skipped[0] += 1
                continue
            except Exception as exc:
                failed[0] += 1
                print("FAILED", (t, s), type(exc).__name__, str(exc)[:300], flush=True)
                continue
            if r is None:
                continue
            done[(t, s)] = r
            if not args.limit and next_slot(t) is not None:
                futs[ex.submit(run_one, t, next_slot(t), URLS[n % len(URLS)])] = (t, next_slot(t))
            tokens[0] += r["completion_tokens"]
            print(f"[{n}] {r['task_id']}#{r['slot']} {r['finish']} grids={r['grids'] is not None} "
                  f"train_ok={r['best_train_ok']}/{r['n_train']} tok={r['completion_tokens']} turns={r['turns']} "
                  f"checks={r['checks']} {r['wall_s']}s | verified {len(VERIFIED)} | queued {len(futs)} | agg "
                  f"{tokens[0] / (time.time() - t_start):.0f} tok/s", flush=True)
print(f"finished: {failed[0]} failed, {skipped[0]} not started, verified tasks {len(VERIFIED)}", flush=True)
sys.exit(1 if failed[0] else 0)


In [ ]:
%%writefile /kaggle/working/ab/a100/v7_merge.py
"""Merge v7 Qwen rows into NVARC's submission (the hybrid6 rule, verified answers first), and score A/B runs.

Rule per output, rows in priority order with verified rows before unverified ones and slot 0 before slot 1:
  - a row is usable when it has a grid and (it is verified, or it finished with "stop", or NVARC has no candidate)
  - a grid equal to either current attempt is skipped
  - NVARC has no candidate (or its attempt_1 is the [[0]] placeholder): first grid -> attempt_1 (attempt_2 = a
    distinct fallback), the next one -> attempt_2
  - NVARC weak (n < 2 or v2 <= v2_max; 2 for 16 views, 1 for 8) and attempt_2 still NVARC's: grid -> attempt_2
  - otherwise (NVARC confident) the output is left alone
attempt_1 is never replaced where NVARC has a candidate, so a wrong Qwen grid cannot cost a point NVARC had in
attempt_1; the loss case is NVARC's attempt_2 being right and replaced (seen once in the hybrid6 A/B, b5ca7ac4).

    python v7_merge.py RUN_DIR --fixed DIR --challenges FILE --solutions FILE     (A/B scoring on public eval)
"""
import json
import sys


def merge(nvarc, conf, rows, order, fallback, select_distinct_fallback, attempt1_policy=None, v2_max=2):
    """Return (final submission, list of (task, output, slot, verified, placed_in)).

    conf: "task_i" -> (n candidates, v1, v2). order: (task, i) -> priority rank. fallback: guaranteed fallback
    submission (task -> list of rows). v2_max: NVARC is weak when n < 2 or v2 <= v2_max
    (2 for 16 views, 1 for 8). attempt1_policy: optional counterfactual for reports only, a function
    (n, v1, v2) -> bool saying a VERIFIED grid may take attempt_1 (NVARC's attempt_1 then moves to attempt_2).
    """
    final = json.loads(json.dumps(nvarc))
    for t in final:
        for i, row in enumerate(final[t]):
            if row["attempt_1"] == row["attempt_2"]:
                row["attempt_2"] = select_distinct_fallback(row["attempt_1"], fallback[t][i])
    base = json.loads(json.dumps(final))
    rows = sorted(rows, key=lambda r: (order.get((r["task_id"], r["query_index"]), 1e9), not r.get("verified"), r["slot"]))
    answered, placed = set(), []
    for r in rows:
        t, i, grid = r["task_id"], r["query_index"], r["grid"]
        if t not in final or i >= len(final[t]) or grid is None:
            continue
        n, v1, v2 = conf.get(f"{t}_{i}", (0, 0, 0))
        if not (r.get("verified") or r["finish"] == "stop" or n == 0):
            continue
        old, nv = final[t][i], base[t][i]
        if grid in (old["attempt_1"], old["attempt_2"]):
            continue
        if n == 0 or nv["attempt_1"] == [[0]]:
            if (t, i) not in answered:
                new, where = {"attempt_1": grid, "attempt_2": select_distinct_fallback(grid, fallback[t][i])}, "a1_empty"
            else:
                new, where = {"attempt_1": old["attempt_1"], "attempt_2": grid}, "a2_empty"
        elif attempt1_policy and r.get("verified") and (t, i) not in answered and attempt1_policy(n, v1, v2):
            new, where = {"attempt_1": grid, "attempt_2": nv["attempt_1"]}, "a1_over_nvarc"
        elif (n < 2 or v2 <= v2_max) and old["attempt_2"] == nv["attempt_2"]:
            new, where = {"attempt_1": old["attempt_1"], "attempt_2": grid}, "a2"
        else:
            continue
        final[t][i] = new
        answered.add((t, i))
        placed.append((t, i, r["slot"], bool(r.get("verified")), where))
    return final, placed


def score(sub, sol, tasks):
    per = {}
    for t in tasks:
        k = len(sol[t])
        per[t] = sum(1 / k for i in range(k) if sol[t][i] in (sub[t][i]["attempt_1"], sub[t][i]["attempt_2"]))
    return per


if __name__ == "__main__":
    import argparse
    import pathlib
    import tempfile
    ap = argparse.ArgumentParser()
    ap.add_argument("run")
    ap.add_argument("--fixed", required=True)
    ap.add_argument("--cohort", default="confirm")
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--solutions", required=True)
    ap.add_argument("--runner-parts", required=True, help="dir with runner_parts.py")
    a = ap.parse_args()
    sys.path.insert(0, a.runner_parts)
    import runner_parts as rp
    AGG = rp.load_module_from_source("v7_agg", pathlib.Path(tempfile.gettempdir()) / "v7_agg.py", rp.AGGREGATOR_SOURCE)
    fx = pathlib.Path(a.fixed)
    raw = json.load(open(a.challenges))
    sol = json.load(open(a.solutions))
    elig = [tuple(x) for x in json.load(open(fx / f"qwen_eligible_{a.cohort}.json"))]
    tasks = list(dict.fromkeys(t for t, _ in elig))
    nv_all = json.load(open(fx / "nvarc_submission.json"))
    nvarc = {t: nv_all[t] for t in tasks}
    conf = {k: tuple(v) for k, v in json.load(open(fx / "nvarc_confidence.json")).items()}
    rows = [json.loads(l) for l in open(pathlib.Path(a.run) / "samples.jsonl")]
    trows = [json.loads(l) for l in open(pathlib.Path(a.run) / "tasks.jsonl")]
    order = {o: n for n, o in enumerate(elig)}
    fallback = AGG.guaranteed_fallback_submission({t: raw[t] for t in tasks})
    s_nv = score(nvarc, sol, tasks)
    print(f"run {a.run}: {len(trows)} episodes over {len(tasks)} tasks, "
          f"{sum(r['completion_tokens'] for r in trows) / 1e6:.2f}M tokens")
    fin = {}
    for r in trows:
        fin[r["finish"]] = fin.get(r["finish"], 0) + 1
    print("finish:", fin)
    # how good are the raw Qwen answers, per kind
    for kind in ("verified", "stop"):
        rs = [r for r in rows if r["grid"] is not None and r["finish"] == kind]
        ok = sum(r["grid"] == sol[r["task_id"]][r["query_index"]] for r in rs)
        print(f"  {kind}: {ok}/{len(rs)} output grids correct")
    policies = {"hybrid rule": None, "verified->a1 if v1<=3": lambda n, v1, v2: v1 <= 3,
                "verified->a1 always": lambda n, v1, v2: True}
    for name, pol in policies.items():
        for slots in (1, 2):
            rr = [r for r in rows if r["slot"] < slots]
            final, placed = merge(nvarc, conf, rr, order, fallback, rp.select_distinct_fallback, pol)
            s = score(final, sol, tasks)
            gain = sum(max(0, s[t] - s_nv[t]) for t in tasks)
            loss = sum(max(0, s_nv[t] - s[t]) for t in tasks)
            print(f"{name:24s} slots<={slots}: NVARC {sum(s_nv.values()):.2f} -> {sum(s.values()):.2f} "
                  f"(gain {gain:.2f}, loss {loss:.2f}, placed {len(placed)}) "
                  f"gained {[t for t in tasks if s[t] > s_nv[t]]} lost {[t for t in tasks if s[t] < s_nv[t]]}")


In [ ]:
!ARC_DECODE_BATCH=8 ARC_MERGE_LORA=1 ARC_N_EVAL_AUG=1 PYTHONHASHSEED=0 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
V2_MAX, V1_TIER0 = 1, 2  # votes are out of 8 views; hybrid6 used 2, 3 with 16

# NVARC is done (or out of time): write its submission (the fallback for everything below), its confidence per output,
# and the outputs Qwen may improve, in priority order (see the builder's docstring).
import json, os
from collections import defaultdict
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon

data_dir = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
CHALLENGES = os.path.join(data_dir, "arc-agi_test_challenges.json" if RERUN else "arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(CHALLENGES)
decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
submission = data.get_submission()
try:
    decoder.load_decoded_results("/kaggle/inference_outputs")
    ArcDataset.fill_submission(decoder.run_selection_algo(), submission)
except Exception as e:
    print("*** NVARC selection failed:", type(e).__name__, e)
json.dump(submission, open("/kaggle/working/nvarc_submission.json", "w"))
json.dump(submission, open("/kaggle/working/submission.json", "w"))

conf = {}   # (task, test index) -> (n unique candidates, votes of attempt_1, votes of attempt_2)
for bk, pool in decoder.decoded_results.items():
    groups = defaultdict(int)
    for s in pool.values():
        groups[tuple(map(tuple, s["solution"]))] += 1
    ranked = score_kgmon(pool)
    votes = [groups[tuple(map(tuple, g))] for g in ranked]
    task, idx = bk.rsplit("_", 1)
    conf[(task, int(idx))] = (len(ranked), votes[0] if votes else 0, votes[1] if len(votes) > 1 else 0)
json.dump({f"{t}_{i}": v for (t, i), v in conf.items()}, open("/kaggle/working/nvarc_confidence.json", "w"))

keys = list(data.keys)
if not RERUN:
    keys = [k for k in keys if k in os.environ["ARC_TASKS"].split(",")]
raw = json.load(open(CHALLENGES))
def size(t):
    return sum(len(g) * len(g[0]) for p in raw[t]["train"] for g in (p["input"], p["output"])) + \
        sum(len(p["input"]) * len(p["input"][0]) for p in raw[t]["test"])
eligible = []
for t in keys:
    for i in range(len(raw[t]["test"])):
        n, v1, v2 = conf.get((t, i), (0, 0, 0))
        if not (n == 0 or n < 2 or v2 <= V2_MAX):
            continue
        tier = 2 if n == 0 else 0 if v1 <= V1_TIER0 else 1
        eligible.append((tier, v1, v2, size(t), t, i))
eligible.sort()
json.dump([[t, i] for *_, t, i in eligible], open("/kaggle/working/ab/live/qwen_eligible_live.json", "w"))
for name in ("nvarc_submission.json", "nvarc_confidence.json"):   # qwen_ab.py fingerprints these
    json.dump(json.load(open("/kaggle/working/" + name)), open("/kaggle/working/ab/live/" + name, "w"))
json.dump({k: raw[k] for k in keys}, open("/kaggle/working/ab/live/challenges.json", "w"))
tiers = defaultdict(int)
for e in eligible:
    tiers[e[0]] += 1
print("outputs:", sum(len(raw[k]["test"]) for k in keys), "| eligible:", len(eligible), "by tier", dict(tiers))
print("first 12:", eligible[:12])
print("NVARC done at", round((time.time() - NOTEBOOK_START) / 3600, 2), "h")


In [ ]:

# vLLM 0.19 (cp311 wheelhouse) into an isolated dir, as in the A/B notebook; the server runs with python -S
import glob, os, subprocess, sys
try:
    wh = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/vllm-0.19.0*.whl", recursive=True)})
    assert len(wh) == 1, wh
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
                        "--find-links", wh[0], "--target", "/tmp/vllm_env", "vllm==0.19.0"], capture_output=True, text=True)
    print("pip", r.returncode, r.stderr[-1500:])
    os.makedirs("/tmp/vllm_env/sklearn", exist_ok=True)   # numpy ABI clash in the NVARC image (see build_hybrid.py)
    open("/tmp/vllm_env/sklearn/__init__.py", "w").write('__version__ = "0.0.0"\n')
    open("/tmp/vllm_env/sklearn/metrics.py", "w").write('def roc_curve(*a, **k):\n    raise RuntimeError("shim")\n')
    cfgs = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "qwen3-8-27b-fp8" in p.lower()]
    assert cfgs, "model not attached"
    MODEL = os.path.dirname(min(cfgs, key=len))
    print("model", MODEL, "| at", round((time.time() - NOTEBOOK_START) / 3600, 2), "h")
except Exception as e:
    MODEL = None
    print("*** vLLM install failed; submission stays NVARC:", type(e).__name__, e)


In [ ]:

# every cell after the handoff must not raise: an exception fails the rerun, and NVARC's submission is already written
SERVED = False
try:
    if MODEL:
        with open("/kaggle/working/serve.log", "w") as f:
            r = subprocess.run(f"cd /kaggle/working/ab && MODEL='{MODEL}' VENV=/tmp/vllm_env TP=4 SEQS=16 KVFP8=1 MTP=1 "
                               "CTX=131072 MEM=0.90 LOGDIR=/kaggle/working/server_logs bash a100/serve.sh", shell=True,
                               stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                               timeout=max(60, QWEN_KILL_AT - 1200 - time.time()))
        print(open("/kaggle/working/serve.log").read()[-3000:])
        SERVED = r.returncode == 0
except Exception as e:
    print("*** serve failed:", type(e).__name__, e)
print("vLLM served:", SERVED, "| at", round((time.time() - NOTEBOOK_START) / 3600, 2), "h")


In [ ]:

# Qwen v7 on the live cohort (tasks in priority order) until QWEN_KILL_AT. Finished episodes are appended to
# samples.jsonl as they complete, so killing the process loses only the episodes still in flight.
import signal
RUN_DIR = "/kaggle/working/runs/live_v7_medium_64k"
try:
  if SERVED and time.time() < QWEN_KILL_AT - 600:
    log = open("/kaggle/working/qwen_live.log", "w")
    p = subprocess.Popen([sys.executable, "a100/qwen_v7.py", "--cohort", "live", "--effort", "medium",
                          "--budget", "65536", "--samples", "2", "--concurrency", "16",
                          "--servers", "/kaggle/working/server_logs/servers.json", "--fixed", "/kaggle/working/ab/live",
                          "--challenges", "/kaggle/working/ab/live/challenges.json", "--out", RUN_DIR,
                          "--stop-launch-at", str(QWEN_KILL_AT - 900)],
                         cwd="/kaggle/working/ab", stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    while p.poll() is None and time.time() < QWEN_KILL_AT:
        time.sleep(30)
    if p.poll() is None:
        print("kill at", round((time.time() - NOTEBOOK_START) / 3600, 2), "h")
        os.killpg(p.pid, signal.SIGKILL)
    log.close()
    print("qwen_v7 exit", p.returncode)
    print("".join(open("/kaggle/working/qwen_live.log").readlines()[-25:]))
  else:
    print("Qwen skipped (server down or no time)")
except Exception as e:
    print("*** Qwen run failed:", type(e).__name__, e)


In [ ]:

# v7_merge.merge: the hybrid6 rule (attempt_2 or empty slots only), verified answers first. Any failure leaves
# NVARC's submission in place.
import json, os, sys, tempfile, pathlib, time
nvarc = json.load(open("/kaggle/working/nvarc_submission.json"))
final = json.loads(json.dumps(nvarc))
try:
    sys.path.insert(0, "/kaggle/working/ab/a100")
    import runner_parts as rp, v7_merge
    AGG = rp.load_module_from_source("qwen38_agg_merge", pathlib.Path(tempfile.gettempdir()) / "qwen38_agg_merge.py", rp.AGGREGATOR_SOURCE)
    raw = json.load(open("/kaggle/working/ab/live/challenges.json"))
    CONF = {k: tuple(v) for k, v in json.load(open("/kaggle/working/nvarc_confidence.json")).items()}
    order = {tuple(x): n for n, x in enumerate(json.load(open("/kaggle/working/ab/live/qwen_eligible_live.json")))}
    path = os.path.join(RUN_DIR, "samples.jsonl")
    rows = [json.loads(l) for l in open(path)] if os.path.exists(path) else []
    merged, placed = v7_merge.merge(nvarc, CONF, rows, order, AGG.guaranteed_fallback_submission(raw),
                                    rp.select_distinct_fallback, v2_max=V2_MAX)
    if set(merged) != set(nvarc) or any(len(merged[t]) != len(nvarc[t]) for t in nvarc):
        raise ValueError("task/output mismatch")
    for t in merged:
        for row in merged[t]:
            for a in ("attempt_1", "attempt_2"):
                AGG.validate_grid(row[a], context="final " + a)
    final = merged
    tpath = os.path.join(RUN_DIR, "tasks.jsonl")
    trows = [json.loads(l) for l in open(tpath)] if os.path.exists(tpath) else []
    fin = {}
    for r in trows:
        fin[r["finish"]] = fin.get(r["finish"], 0) + 1
    print(f"Qwen episodes: {len(trows)} {fin} | grids placed: {len(placed)} "
          f"(verified {sum(p[3] for p in placed)}) {placed[:20]}")
except Exception as e:
    print("*** merge failed, using NVARC:", type(e).__name__, e)
    final = nvarc
json.dump(final, open("/kaggle/working/submission.json", "w"))
print("outputs changed by Qwen:", sum(1 for t in nvarc for i, r in enumerate(nvarc[t]) if final[t][i] != r),
      "| done at", round((time.time() - NOTEBOOK_START) / 3600, 2), "h")
if not RERUN:
    sol = json.load(open("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"))
    keys = os.environ["ARC_TASKS"].split(",")
    score = lambda sub: sum(1 / len(sol[k]) for k in keys for i in range(len(sol[k])) if sol[k][i] in (sub[k][i]["attempt_1"], sub[k][i]["attempt_2"]))
    print(f"on {len(keys)} commit tasks: NVARC {score(nvarc):.2f}  ->  hybrid7 {score(final):.2f}")
